# Champion + counsel + JUDGE inference pipeline
This notebook is extracted from the supplied CS781 notebook. It preserves the original implementation and skips cells 362–363 (training experiments).

Before running:
- Confirm the Kaggle datasets are attached.
- This pipeline uses the original `champion_recovery.pt` and its thresholds.
- It writes a new candidate only if the destination does not already exist.
- Review the fold-2 score before accepting the fold-3 output.


In [ ]:

import torch
import torch.nn as nn
from transformers import GPT2Config, GPT2Model
import json
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_artifact(filename, preferred_parts=()):
    roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(
                p for p in root.rglob(filename) if p.is_file()
            )

    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(
            f"Cannot find {filename} under /kaggle/input or "
            "/kaggle/working. Attach or download the dataset containing it."
        )

    preferred = [
        p for p in matches
        if all(part in str(p) for part in preferred_parts)
    ]
    chosen = preferred[0] if preferred else matches[0]
    print(f"{filename} -> {chosen}")
    return chosen

RECOVERY_PATH = find_artifact(
    "champion_recovery.pt", ("cs781_recovery",)
)
PHASE1_PATH = find_artifact(
    "checkpoint_latest.pt", ("cs781-phase-1-checkpoints",)
)
THRESHOLDS_PATH = find_artifact(
    "champion_thresholds_recovery.json", ("cs781_recovery",)
)

recovery = torch.load(
    RECOVERY_PATH, map_location="cpu", weights_only=False
)
phase1_ckpt = torch.load(
    PHASE1_PATH, map_location="cpu", weights_only=False
)

BIO_LABELS = recovery["bio_labels"]
label2id = {label: i for i, label in enumerate(BIO_LABELS)}
id2label = {i: label for i, label in enumerate(BIO_LABELS)}

cfg = phase1_ckpt.get("model_config")
if cfg is None:
    raise KeyError("Phase 1 checkpoint has no model_config.")

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

class LegalNERModel(nn.Module):
    def __init__(self, pretrained_lm, num_labels):
        super().__init__()
        self.encoder = pretrained_lm.transformer
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(
            pretrained_lm.config.hidden_size, num_labels
        )

    def forward(self, input_ids, attention_mask=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden = self.dropout(outputs.last_hidden_state)
        return self.classifier(hidden)

# Reconstruct the Phase 1 GPT-2 encoder.
phase1_lm = GPT2Model(gpt2_config)
phase1_state = {
    k.removeprefix("transformer."): v
    for k, v in phase1_ckpt["model_state_dict"].items()
    if k.startswith("transformer.")
}
phase1_lm.load_state_dict(phase1_state, strict=True)

# Build NER architecture and load the trained recovery weights.
lm_wrapper = type("LMWrapper", (), {
    "transformer": phase1_lm,
    "config": gpt2_config
})()

champion_model = LegalNERModel(
    pretrained_lm=lm_wrapper,
    num_labels=len(BIO_LABELS)
).to(device)

load_result = champion_model.load_state_dict(
    recovery["model_state_dict"], strict=True
)
champion_model.eval()
ner_model = champion_model

with open(THRESHOLDS_PATH, "r", encoding="utf-8") as f:
    champion_thresholds_fusion = json.load(f)

print("Checkpoint load:", load_result)
print("Device:", device)
print("BIO labels:", len(BIO_LABELS))
print("Threshold data type:", type(champion_thresholds_fusion).__name__)

# Shape check only; this does not evaluate model quality.
with torch.inference_mode():
    test_ids = torch.zeros((1, 16), dtype=torch.long, device=device)
    test_mask = torch.ones_like(test_ids)
    test_logits = champion_model(test_ids, test_mask)

assert test_logits.shape == (1, 16, len(BIO_LABELS))
print("Test logits shape:", tuple(test_logits.shape))
print("Champion loaded and shape check passed.")


In [ ]:

import torch

recovery_state = recovery["model_state_dict"]
loaded_state = champion_model.state_dict()

assert set(recovery_state.keys()) == set(loaded_state.keys())

max_param_diff = max(
    (loaded_state[name].detach().cpu() - recovery_state[name]).abs().max().item()
    for name in loaded_state
)

champion_model.eval()
with torch.inference_mode():
    x = torch.zeros((1, 16), dtype=torch.long, device=device)
    mask = torch.ones_like(x)

    logits1 = champion_model(x, mask)
    logits2 = champion_model(x, mask)

print("Maximum parameter difference:", max_param_diff)
print("Logits shape:", tuple(logits1.shape))
print("Repeated-inference difference:", (logits1 - logits2).abs().max().item())

assert max_param_diff == 0.0
assert torch.allclose(logits1, logits2, atol=1e-6)

print("Recovery verification passed.")


In [ ]:

from pathlib import Path
import json
from tokenizers import Tokenizer

def find_file(filename):
    roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(
                p for p in root.rglob(filename) if p.is_file()
            )

    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found. Check the attached Kaggle dataset."
        )

    print(f"\nFound {filename}:")
    for p in matches:
        print(" ", p)
    return matches[0]

tokenizer_path = find_file("legal_bpe_32k.json")
fold1_text_path = find_file("fold1-text.json")
fold1_gold_path = find_file("fold1-gold.json")
fold2_text_path = find_file("fold2-text.json")
fold2_gold_path = find_file("fold2-gold.json")
fold3_text_path = find_file("fold3-text.json")

tokenizer = Tokenizer.from_file(str(tokenizer_path))

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_texts = load_json(fold1_text_path)
train_gold = load_json(fold1_gold_path)
val_texts = load_json(fold2_text_path)
val_gold = load_json(fold2_gold_path)
test_texts = load_json(fold3_text_path)

print("\nTokenizer and datasets loaded successfully.")
print("Train:", len(train_texts), "documents")
print("Validation:", len(val_texts), "documents")
print("Test:", len(test_texts), "documents")


In [ ]:

import numpy as np
from collections import Counter, defaultdict
from tqdm.auto import tqdm

champion_model.eval()

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]
MAX_WINDOW = 1024
WINDOW_STRIDE_DIAG = 768

assert len(BIO_LABELS) == 25
assert all(f"B-{lab}" in label2id for lab in WEAK_LABELS)

# Accumulate logits over overlapping windows, one window at a time.
doc_logits = {}
with torch.inference_mode():
    for doc_id, text in tqdm(val_texts.items(), desc="Scoring validation"):
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids = enc.ids
        n = len(ids)
        if n == 0:
            doc_logits[doc_id] = (enc.offsets, np.zeros((0, len(BIO_LABELS))))
            continue

        sums = np.zeros((n, len(BIO_LABELS)), dtype=np.float32)
        counts = np.zeros(n, dtype=np.float32)

        starts = list(range(0, n, WINDOW_STRIDE_DIAG))
        for start in starts:
            end = min(start + MAX_WINDOW, n)
            if start >= n:
                break

            x = torch.tensor(
                [ids[start:end]], dtype=torch.long, device=device
            )
            mask = torch.ones_like(x)
            logits = champion_model(x, mask)[0].float().cpu().numpy()

            sums[start:end] += logits
            counts[start:end] += 1

            if end == n:
                break

        avg_logits = sums / np.maximum(counts[:, None], 1)
        doc_logits[doc_id] = (enc.offsets, avg_logits)

# Summarize token-level predictions inside annotated weak-label spans.
top_counts = {lab: Counter() for lab in WEAK_LABELS}
target_probs = defaultdict(list)
entity_count = Counter()
token_count = Counter()

for doc_id, text in val_texts.items():
    offsets, logits = doc_logits[doc_id]
    if len(offsets) == 0:
        continue

    # Stable softmax over BIO classes.
    shifted = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    probs /= probs.sum(axis=1, keepdims=True)
    pred_ids = logits.argmax(axis=1)

    for ann in val_gold[doc_id]:
        start, end, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in WEAK_LABELS:
            continue

        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]
        if not token_indices:
            continue

        entity_count[label] += 1
        target_probs[label].append(
            float(probs[token_indices, label2id[f"B-{label}"]].mean())
        )

        for i in token_indices:
            token_count[label] += 1
            top_counts[label][BIO_LABELS[int(pred_ids[i])]] += 1

print("\nWEAK-LABEL VALIDATION DIAGNOSTIC")
for lab in WEAK_LABELS:
    mean_prob = (
        float(np.mean(target_probs[lab])) if target_probs[lab] else float("nan")
    )
    print(f"\n{lab}: {entity_count[lab]} gold entities")
    print(f"Mean token probability for B-{lab}: {mean_prob:.4f}")
    print(f"Gold-span tokens: {token_count[lab]}")
    print("Top predicted BIO labels:", top_counts[lab].most_common(8))


In [ ]:

import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print("Device:", torch.cuda.get_device_name(0))
print("GPU allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GiB")
print("GPU reserved:", round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GiB")
print("GPU total:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GiB")

print("\nChampion training mode:", champion_model.training)
print("Champion parameters:", sum(p.numel() for p in champion_model.parameters()))
print("Champion dtype:", next(champion_model.parameters()).dtype)
print("Champion device:", next(champion_model.parameters()).device)

print("\nExisting saved recovery:", 
      __import__("os").path.exists("/kaggle/working/cs781_backup_download/cs781_recovery/champion_recovery.pt"))


In [ ]:

import copy
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

# Keep an independent CPU reference for exact recovery.
champion_reference_cpu = {
    k: v.detach().cpu().clone()
    for k, v in champion_model.state_dict().items()
}

# Train a separate copy; leave champion_model untouched.
targeted_model = copy.deepcopy(champion_model).to(device)
targeted_model.train()

# Only AdamW state for trainable parameters will be created lazily
# during the first optimizer step.
targeted_optimizer = torch.optim.AdamW(
    targeted_model.parameters(),
    lr=1e-6,
    weight_decay=0.01
)

# Conservative weak-label weighting.
weak_labels = {"RESP", "A.COUNSEL", "R.COUNSEL", "WIT"}
class_weights = torch.ones(len(BIO_LABELS), dtype=torch.float32, device=device)
class_weights[label2id["O"]] = 0.5

for lab in weak_labels:
    class_weights[label2id[f"B-{lab}"]] = 4.0
    class_weights[label2id[f"I-{lab}"]] = 4.0

targeted_criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=-100
)

print("Separate training copy created.")
print("Champion remains in eval mode:", not champion_model.training)
print("Targeted copy in training mode:", targeted_model.training)
print("Learning rate:", targeted_optimizer.param_groups[0]["lr"])
print("GPU allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GiB")
print("Champion recovery reference tensors:", len(champion_reference_cpu))


In [ ]:

import random
from collections import Counter

MAX_LEN_PILOT = 256
STRIDE_PILOT = 192
IGNORE_INDEX = -100

def make_pilot_windows(texts, gold, tokenizer,
                       max_len=MAX_LEN_PILOT, stride=STRIDE_PILOT):
    windows = []
    label_counts = Counter()

    for doc_id, text in texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids, offsets = enc.ids, enc.offsets
        if not ids:
            continue

        token_labels = ["O"] * len(ids)

        # Assign BIO labels from gold character spans.
        for ann in gold.get(doc_id, []):
            start, end, label = int(ann[0]), int(ann[1]), ann[2]
            overlapping = [
                i for i, (a, b) in enumerate(offsets)
                if b > a and a < end and b > start
            ]
            if not overlapping or label not in weak_labels and label not in {
                "APP", "JUDGE", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
            }:
                continue

            for j, tok_i in enumerate(overlapping):
                token_labels[tok_i] = ("B-" if j == 0 else "I-") + label

        start = 0
        while start < len(ids):
            end = min(start + max_len, len(ids))
            win_ids = ids[start:end]
            win_tags = token_labels[start:end]

            # Avoid duplicate supervision in overlapping context.
            supervised_start = 0 if start == 0 else max(0, max_len - stride)
            train_tags = [
                label2id[tag] if i >= supervised_start else IGNORE_INDEX
                for i, tag in enumerate(win_tags)
            ]

            if any(tag != IGNORE_INDEX for tag in train_tags):
                windows.append({
                    "input_ids": win_ids,
                    "labels": train_tags
                })
                label_counts.update(
                    BIO_LABELS[tag] for tag in train_tags if tag != IGNORE_INDEX
                )

            if end == len(ids):
                break
            start += stride

    return windows, label_counts

pilot_windows, pilot_label_counts = make_pilot_windows(
    train_texts, train_gold, tokenizer
)

print("Pilot windows:", len(pilot_windows))
print("Supervised tokens:", sum(pilot_label_counts.values()))
print("Weak-label supervised token counts:")
for lab in ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]:
    print(
        lab,
        pilot_label_counts[f"B-{lab}"] + pilot_label_counts[f"I-{lab}"]
    )

print("\nExample window length:", len(pilot_windows[0]["input_ids"]))
print("Example supervised labels:", sum(
    x != IGNORE_INDEX for x in pilot_windows[0]["labels"]
))


In [ ]:

from collections import Counter

audit = Counter()
bad_examples = []

for doc_id, text in train_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    offsets = enc.offsets
    annotations = sorted(
        train_gold.get(doc_id, []),
        key=lambda x: (int(x[0]), int(x[1]))
    )

    previous_end = -1

    for ann in annotations:
        start, end, label = int(ann[0]), int(ann[1]), ann[2]

        if start < previous_end:
            audit["overlapping_annotations"] += 1
            if len(bad_examples) < 5:
                bad_examples.append((doc_id, "overlap", ann))

        previous_end = max(previous_end, end)

        if start < 0 or end > len(text) or start >= end:
            audit["invalid_character_span"] += 1
            continue

        covered = [
            (a, b) for a, b in offsets
            if b > a and a < end and b > start
        ]

        if not covered:
            audit["no_token_coverage"] += 1
            continue

        if covered[0][0] != start or covered[-1][1] != end:
            audit["boundary_not_token_aligned"] += 1
            if len(bad_examples) < 5:
                bad_examples.append(
                    (doc_id, "boundary", (start, end, label, covered[0], covered[-1]))
                )
        else:
            audit["exactly_token_aligned"] += 1

print("Training annotation audit:")
for key in [
    "overlapping_annotations",
    "invalid_character_span",
    "no_token_coverage",
    "boundary_not_token_aligned",
    "exactly_token_aligned"
]:
    print(f"{key}: {audit[key]}")

print("\nExamples of issues:")
for item in bad_examples:
    print(item)


In [ ]:

from collections import defaultdict

examples_by_label = defaultdict(list)

for doc_id, text in train_texts.items():
    offsets = tokenizer.encode(text, add_special_tokens=False).offsets

    for ann in train_gold.get(doc_id, []):
        start, end, label = int(ann[0]), int(ann[1]), ann[2]

        if label not in {"RESP", "A.COUNSEL", "R.COUNSEL", "WIT"}:
            continue

        covered = [
            (a, b) for a, b in offsets
            if b > a and a < end and b > start
        ]
        if not covered:
            continue

        if covered[0][0] != start or covered[-1][1] != end:
            if len(examples_by_label[label]) < 4:
                token_start, token_end = covered[0][0], covered[-1][1]
                examples_by_label[label].append({
                    "doc_id": doc_id,
                    "annotation": (start, end, label),
                    "annotated_text": repr(text[start:end]),
                    "token_covered_text": repr(text[token_start:token_end]),
                    "left_difference": repr(text[min(start, token_start):max(start, token_start)]),
                    "right_difference": repr(text[min(end, token_end):max(end, token_end)])
                })

for label in ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]:
    print(f"\n--- {label} ---")
    for ex in examples_by_label[label]:
        for key, value in ex.items():
            print(f"{key}: {value}")
        print()


In [ ]:

from collections import Counter, defaultdict

weak_labels = {"RESP", "A.COUNSEL", "R.COUNSEL", "WIT"}
boundary_stats = defaultdict(Counter)
boundary_examples = defaultdict(list)

for doc_id, text in train_texts.items():
    offsets = tokenizer.encode(text, add_special_tokens=False).offsets

    for ann in train_gold.get(doc_id, []):
        start, end, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in weak_labels:
            continue

        covered = [
            (a, b) for a, b in offsets
            if b > a and a < end and b > start
        ]
        if not covered:
            boundary_stats[label]["no_token_coverage"] += 1
            continue

        first_start, last_end = covered[0][0], covered[-1][1]

        left_extra = max(0, start - first_start)
        right_extra = max(0, last_end - end)

        if left_extra == 0 and right_extra == 0:
            boundary_stats[label]["exact"] += 1
        elif (
            text[first_start:start].strip() == ""
            and text[end:last_end].strip() == ""
        ):
            boundary_stats[label]["whitespace_only"] += 1
        else:
            boundary_stats[label]["non_whitespace_mismatch"] += 1
            if len(boundary_examples[label]) < 3:
                boundary_examples[label].append({
                    "doc_id": doc_id,
                    "span": (start, end),
                    "entity": repr(text[start:end]),
                    "token_covered": repr(text[first_start:last_end]),
                    "left_extra": repr(text[first_start:start]),
                    "right_extra": repr(text[end:last_end])
                })

print("Weak entity boundary audit:")
for lab in sorted(weak_labels):
    print(f"\n{lab}: {dict(boundary_stats[lab])}")
    for ex in boundary_examples[lab]:
        print(ex)


In [ ]:

from collections import Counter

MAX_LEN_PILOT = 256
STRIDE_PILOT = 192
IGNORE_INDEX = -100

def make_unique_pilot_windows(texts, gold, tokenizer,
                              max_len=MAX_LEN_PILOT, stride=STRIDE_PILOT):
    windows = []
    label_counts = Counter()

    for doc_id, text in texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids, offsets = enc.ids, enc.offsets
        n = len(ids)
        if n == 0:
            continue

        tags = ["O"] * n

        # Assign each annotation to the tokens overlapping its character span.
        # Gold annotations do not overlap, as verified by the earlier audit.
        for ann in gold.get(doc_id, []):
            start, end, label = int(ann[0]), int(ann[1]), ann[2]
            token_indices = [
                i for i, (a, b) in enumerate(offsets)
                if b > a and a < end and b > start
            ]
            if not token_indices:
                continue

            for j, i in enumerate(token_indices):
                tags[i] = ("B-" if j == 0 else "I-") + label

        # Each token is loss-supervised exactly once: the window in which
        # that token is first encountered owns its label.
        supervised = [False] * n
        start = 0

        while start < n:
            end = min(start + max_len, n)
            win_ids = ids[start:end]
            win_labels = []

            for global_i in range(start, end):
                if not supervised[global_i]:
                    win_labels.append(label2id[tags[global_i]])
                    supervised[global_i] = True
                    label_counts[tags[global_i]] += 1
                else:
                    win_labels.append(IGNORE_INDEX)

            if any(x != IGNORE_INDEX for x in win_labels):
                windows.append({
                    "doc_id": doc_id,
                    "start": start,
                    "input_ids": win_ids,
                    "labels": win_labels
                })

            if end == n:
                break
            start += stride

        assert all(supervised), f"Some tokens were not assigned in {doc_id}"

    return windows, label_counts

pilot_windows, pilot_label_counts = make_unique_pilot_windows(
    train_texts, train_gold, tokenizer
)

print("Corrected windows:", len(pilot_windows))
print("Uniquely supervised tokens:", sum(pilot_label_counts.values()))
print("Label counts:")
for lab in ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]:
    print(lab, pilot_label_counts[f"B-{lab}"] + pilot_label_counts[f"I-{lab}"])

print("First window length:", len(pilot_windows[0]["input_ids"]))
print("First window supervised tokens:",
      sum(x != IGNORE_INDEX for x in pilot_windows[0]["labels"]))


In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
import gc

class PilotNERDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx]

def pilot_collate(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_masks, labels = [], [], []

    for item in batch:
        n = len(item["input_ids"])
        pad = max_len - n
        input_ids.append(item["input_ids"] + [0] * pad)
        attention_masks.append([1] * n + [0] * pad)
        labels.append(item["labels"] + [IGNORE_INDEX] * pad)

    return (
        torch.tensor(input_ids, dtype=torch.long),
        torch.tensor(attention_masks, dtype=torch.long),
        torch.tensor(labels, dtype=torch.long)
    )

pilot_loader = DataLoader(
    PilotNERDataset(pilot_windows),
    batch_size=1,
    shuffle=True,
    collate_fn=pilot_collate
)

# Verify one forward/backward pass only. No optimizer step yet.
targeted_model.train()
targeted_optimizer.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()

input_ids, attention_mask, labels = next(iter(pilot_loader))
input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)
labels = labels.to(device)

try:
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = targeted_model(input_ids, attention_mask)
        loss = targeted_criterion(
            logits.reshape(-1, len(BIO_LABELS)),
            labels.reshape(-1)
        )

    print("Batch shape:", tuple(input_ids.shape))
    print("Loss finite:", bool(torch.isfinite(loss).item()))
    print("Loss:", float(loss.detach().cpu()))
    print("Logits shape:", tuple(logits.shape))

    loss.backward()
    print("Backward pass: successful")
    print("GPU allocated after backward:",
          round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GiB")
finally:
    targeted_optimizer.zero_grad(set_to_none=True)
    del input_ids, attention_mask, labels
    if "logits" in locals():
        del logits
    if "loss" in locals():
        del loss
    gc.collect()
    torch.cuda.empty_cache()

print("Safety check complete; no optimizer step was taken.")


In [ ]:

import os
import gc
import math
import torch
from tqdm.auto import tqdm

MAX_OPTIMIZER_STEPS = 8
GRAD_ACCUM_STEPS = 2
PILOT_SAVE_DIR = "/kaggle/working/phase2_ner_targeted_pilot"
os.makedirs(PILOT_SAVE_DIR, exist_ok=True)

targeted_model.train()
targeted_optimizer.zero_grad(set_to_none=True)

# Mixed-precision gradient scaling for the T4.
scaler = torch.amp.GradScaler("cuda")
loss_history = []
optimizer_steps = 0
micro_steps = 0
pilot_failed = False

try:
    iterator = iter(pilot_loader)

    while optimizer_steps < MAX_OPTIMIZER_STEPS:
        targeted_optimizer.zero_grad(set_to_none=True)
        accumulated_losses = []

        for _ in range(GRAD_ACCUM_STEPS):
            try:
                batch = next(iterator)
            except StopIteration:
                iterator = iter(pilot_loader)
                batch = next(iterator)

            input_ids, attention_mask, labels = [
                tensor.to(device) for tensor in batch
            ]

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = targeted_model(input_ids, attention_mask)
                loss = targeted_criterion(
                    logits.reshape(-1, len(BIO_LABELS)),
                    labels.reshape(-1)
                )
                scaled_loss = loss / GRAD_ACCUM_STEPS

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss; stopping pilot.")

            scaler.scale(scaled_loss).backward()
            accumulated_losses.append(float(loss.detach().cpu()))
            micro_steps += 1

            del input_ids, attention_mask, labels, logits, loss, scaled_loss

        scaler.unscale_(targeted_optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            targeted_model.parameters(), max_norm=1.0
        )

        if not torch.isfinite(grad_norm):
            raise RuntimeError("Non-finite gradient norm; stopping pilot.")

        scaler.step(targeted_optimizer)
        scaler.update()

        optimizer_steps += 1
        mean_loss = sum(accumulated_losses) / len(accumulated_losses)
        loss_history.append(mean_loss)

        print(
            f"Step {optimizer_steps}/{MAX_OPTIMIZER_STEPS} | "
            f"mean loss={mean_loss:.4f} | "
            f"grad norm={float(grad_norm):.3f} | "
            f"GPU allocated={torch.cuda.memory_allocated(0)/1024**3:.2f} GiB"
        )

except Exception as e:
    pilot_failed = True
    print("PILOT STOPPED:", repr(e))

finally:
    targeted_optimizer.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()

if not pilot_failed and optimizer_steps == MAX_OPTIMIZER_STEPS:
    pilot_path = os.path.join(PILOT_SAVE_DIR, "targeted_pilot_8_steps.pt")
    torch.save({
        "model_state_dict": {
            k: v.detach().cpu() for k, v in targeted_model.state_dict().items()
        },
        "optimizer_steps": optimizer_steps,
        "loss_history": loss_history,
        "bio_labels": BIO_LABELS,
    }, pilot_path)
    print("\nPilot checkpoint saved separately:", pilot_path)
else:
    print("\nNo completed-pilot checkpoint saved.")

print("Completed optimizer steps:", optimizer_steps)
print("Loss history:", [round(x, 4) for x in loss_history])
print("Original champion unchanged in memory:", not champion_model.training)


In [ ]:

import torch
import numpy as np

ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]

# Original notebook's span decoding: trim whitespace, preserve exact offsets.
def decode_spans(text, offsets, pred_ids):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    def close():
        nonlocal current_label, current_start, current_end
        if current_label is not None:
            start, end = current_start, current_end
            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1
            if start < end:
                spans.append((start, end, current_label))
        current_label = current_start = current_end = None

    for (start, end), pred_id in zip(offsets, pred_ids):
        if end <= start:
            continue
        tag = BIO_LABELS[int(pred_id)]

        if tag == "O":
            close()
            continue

        prefix, label = tag.split("-", 1)
        if prefix == "B" or current_label != label:
            close()
            current_label, current_start, current_end = label, start, end
        else:
            current_end = end

    close()
    return sorted(set(spans))


# Original notebook's confidence-aware prediction path.
def predict_doc_constrained_with_confidence(text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets

    if not ids:
        return []

    logit_sum = torch.zeros(
        (len(ids), len(BIO_LABELS)), dtype=torch.float32
    )
    token_count = torch.zeros(len(ids), dtype=torch.float32)

    ner_model.eval()
    with torch.no_grad():
        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            chunk = ids[start:end]

            input_ids = torch.tensor(
                [chunk], dtype=torch.long, device=device
            )
            attention_mask = torch.ones_like(input_ids)
            output = ner_model(input_ids, attention_mask)
            logits = output[0] if isinstance(output, tuple) else output
            logits = logits[0].detach().float().cpu()

            chunk_len = end - start
            logit_sum[start:end] += logits[:chunk_len]
            token_count[start:end] += 1

            if end == len(ids):
                break

    averaged_logits = logit_sum / token_count.clamp_min(1).unsqueeze(-1)
    probabilities = torch.softmax(averaged_logits, dim=-1)

    pred_ids = bio_viterbi_decode(averaged_logits)
    spans = decode_spans(text, offsets, pred_ids)
    pred_ids_tensor = torch.tensor(pred_ids, dtype=torch.long)

    results = []
    for start, end, label in spans:
        token_indices = [
            i for i, (s, e) in enumerate(offsets)
            if e > s and s < end and e > start
        ]
        confidence = (
            float(probabilities[
                token_indices, pred_ids_tensor[token_indices]
            ].mean().item())
            if token_indices else 0.0
        )
        results.append((start, end, label, confidence))

    return results


# Original notebook's threshold filter.
def filter_predictions_by_thresholds_for_texts(predictions, thresholds, texts):
    filtered = {}
    for doc_id, spans in predictions.items():
        text = texts[doc_id]
        kept = []
        for s, e, label, conf in spans:
            if conf >= thresholds.get(label, 1.0):
                if 0 <= s < e <= len(text):
                    kept.append((int(s), int(e), label))
        filtered[doc_id] = kept
    return filtered


# Original notebook's exact-span metric implementation.
def calculate_exact_span_metrics(gold_by_doc, pred_by_doc, labels):
    metrics = {}
    for label in labels:
        gold_set, pred_set = set(), set()

        for doc_id, entities in gold_by_doc.items():
            for entity in entities:
                s, e, lab = int(entity[0]), int(entity[1]), str(entity[2])
                if lab == label:
                    gold_set.add((str(doc_id), s, e, lab))

        for doc_id, spans in pred_by_doc.items():
            for span in spans:
                s, e, lab = int(span[0]), int(span[1]), str(span[2])
                if lab == label:
                    pred_set.add((str(doc_id), s, e, lab))

        tp = len(gold_set & pred_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        metrics[label] = {
            "gold": len(gold_set), "pred": len(pred_set),
            "TP": tp, "FP": fp, "FN": fn,
            "precision": precision, "recall": recall, "f1": f1
        }

    metrics["MACRO_F1"] = sum(
        metrics[label]["f1"] for label in labels
    ) / len(labels)
    return metrics


print("Original notebook helper functions restored.")
print("Viterbi function still needs restoration before evaluation.")


In [ ]:

import numpy as np
import torch

NUM_LABELS = len(BIO_LABELS)

# Valid sequence starts: O or any B-entity tag.
legal_start = np.array([
    tag == "O" or tag.startswith("B-")
    for tag in BIO_LABELS
], dtype=bool)

# A transition into I-X is legal only from B-X or I-X.
legal_transition = np.ones(
    (NUM_LABELS, NUM_LABELS), dtype=bool
)

for prev_id, prev_tag in enumerate(BIO_LABELS):
    for curr_id, curr_tag in enumerate(BIO_LABELS):
        if curr_tag.startswith("I-"):
            entity = curr_tag[2:]
            legal_transition[prev_id, curr_id] = (
                prev_tag == f"B-{entity}" or
                prev_tag == f"I-{entity}"
            )

def bio_viterbi_decode(logits):
    """BIO-constrained Viterbi; accepts a torch tensor or NumPy array."""
    emissions = (
        logits.detach().float().cpu().numpy()
        if isinstance(logits, torch.Tensor)
        else np.asarray(logits, dtype=np.float64)
    )
    if emissions.ndim != 2 or emissions.shape[1] != NUM_LABELS:
        raise ValueError(f"Expected [sequence_length, {NUM_LABELS}] logits")
    if len(emissions) == 0:
        return []

    emissions = emissions.astype(np.float64, copy=False)
    neg_inf = -1e12

    scores = np.full_like(emissions, neg_inf, dtype=np.float64)
    backpointers = np.zeros(emissions.shape, dtype=np.int32)

    scores[0, legal_start] = emissions[0, legal_start]

    for t in range(1, len(emissions)):
        transition_scores = scores[t - 1][:, None]
        transition_scores = np.where(
            legal_transition, transition_scores, neg_inf
        )
        candidates = transition_scores + emissions[t][None, :]
        backpointers[t] = np.argmax(candidates, axis=0)
        scores[t] = candidates[
            backpointers[t], np.arange(NUM_LABELS)
        ]

    last = int(np.argmax(scores[-1]))
    path = [last]
    for t in range(len(emissions) - 1, 0, -1):
        path.append(int(backpointers[t, path[-1]]))
    path.reverse()
    return path


# Sanity-check starts and transitions with a deliberately invalid raw path.
test_logits = np.zeros((3, NUM_LABELS), dtype=np.float32)
test_logits[0, label2id["I-RESP"]] = 10.0
test_logits[0, label2id["B-RESP"]] = 8.0
test_logits[1, label2id["I-RESP"]] = 10.0
test_logits[2, label2id["I-WIT"]] = 10.0
test_logits[2, label2id["O"]] = 1.0

test_path = bio_viterbi_decode(test_logits)
test_tags = [BIO_LABELS[i] for i in test_path]

assert not test_tags[0].startswith("I-")
for i, tag in enumerate(test_tags):
    if tag.startswith("I-"):
        assert i > 0
        assert test_tags[i - 1] in (f"B-{tag[2:]}", f"I-{tag[2:]}")

print("Legal starts:", int(legal_start.sum()), "of", NUM_LABELS)
print("Legal transitions:", int(legal_transition.sum()))
print("Sanity-test decoded tags:", test_tags)
print("BIO constraint sanity check passed.")


In [ ]:

import numpy as np
import torch
from tqdm.auto import tqdm

EVAL_MAX_LEN = 1024
EVAL_STRIDE = 768

def predict_with_model_and_confidence(model, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    n = len(ids)

    if n == 0:
        return []

    logit_sum = torch.zeros((n, len(BIO_LABELS)), dtype=torch.float32)
    token_count = torch.zeros(n, dtype=torch.float32)

    model.eval()
    with torch.inference_mode():
        for start in range(0, n, EVAL_STRIDE):
            end = min(start + EVAL_MAX_LEN, n)
            x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
            mask = torch.ones_like(x)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(x, mask)[0].float()

            chunk_logits = logits.cpu()
            logit_sum[start:end] += chunk_logits
            token_count[start:end] += 1

            if end == n:
                break

    avg_logits = logit_sum / token_count.clamp_min(1).unsqueeze(-1)
    probs = torch.softmax(avg_logits, dim=-1).numpy()
    pred_ids = bio_viterbi_decode(avg_logits)
    spans = decode_spans(text, offsets, pred_ids)

    results = []
    for start, end, label in spans:
        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]
        confidence = (
            float(np.mean([
                probs[i, pred_ids[i]] for i in token_indices
            ]))
            if token_indices else 0.0
        )
        results.append((start, end, label, confidence))

    return results


def evaluate_model(model, texts, gold, thresholds):
    raw_predictions = {}
    for doc_id, text in tqdm(texts.items(), desc="Evaluating"):
        raw_predictions[doc_id] = predict_with_model_and_confidence(model, text)

    filtered_predictions = filter_predictions_by_thresholds_for_texts(
        raw_predictions, thresholds, texts
    )
    metrics = calculate_exact_span_metrics(
        gold, filtered_predictions, ENTITY_LABELS
    )
    return metrics, raw_predictions, filtered_predictions


# Make sure threshold values are numeric.
eval_thresholds = {
    label: float(champion_thresholds_fusion.get(label, 1.0))
    for label in ENTITY_LABELS
}
print("Thresholds used:", eval_thresholds)

champion_val_metrics, champion_val_raw, champion_val_pred = evaluate_model(
    champion_model, val_texts, val_gold, eval_thresholds
)

pilot_val_metrics, pilot_val_raw, pilot_val_pred = evaluate_model(
    targeted_model, val_texts, val_gold, eval_thresholds
)

print("\n=== Fold-2 exact-span results ===")
print(f"Champion macro-F1: {champion_val_metrics['MACRO_F1']:.6f}")
print(f"Pilot macro-F1:    {pilot_val_metrics['MACRO_F1']:.6f}")
print(f"Difference:        {pilot_val_metrics['MACRO_F1'] - champion_val_metrics['MACRO_F1']:+.6f}")

print("\nPer-label F1:")
print(f"{'Label':<12} {'Champion':>10} {'Pilot':>10} {'Delta':>10}")
for label in ENTITY_LABELS:
    c = champion_val_metrics[label]["f1"]
    p = pilot_val_metrics[label]["f1"]
    print(f"{label:<12} {c:10.4f} {p:10.4f} {p-c:+10.4f}")

print("\nPredicted span counts:",
      sum(map(len, champion_val_pred.values())),
      sum(map(len, pilot_val_pred.values())))


In [ ]:

import numpy as np
from collections import defaultdict

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

def weak_proposal_report(raw_predictions, gold, thresholds):
    report = {}

    for label in WEAK_LABELS:
        gold_set = set()
        pred_set = set()
        confidences = []

        for doc_id, anns in gold.items():
            for ann in anns:
                if ann[2] == label:
                    gold_set.add((str(doc_id), int(ann[0]), int(ann[1]), label))

        for doc_id, spans in raw_predictions.items():
            for span in spans:
                s, e, lab, conf = span
                if lab == label:
                    pred_set.add((str(doc_id), int(s), int(e), label))
                    confidences.append(float(conf))

        tp = len(gold_set & pred_set)
        precision = tp / len(pred_set) if pred_set else 0.0
        recall = tp / len(gold_set) if gold_set else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        report[label] = {
            "gold": len(gold_set),
            "raw_predictions": len(pred_set),
            "exact_matches": tp,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "threshold": float(thresholds.get(label, 1.0)),
            "confidences": confidences,
        }

    return report

champion_weak_report = weak_proposal_report(
    champion_val_raw, val_gold, eval_thresholds
)
pilot_weak_report = weak_proposal_report(
    pilot_val_raw, val_gold, eval_thresholds
)

print(f"{'Label':<12} {'Model':<10} {'Gold':>6} {'Raw':>6} {'Exact':>6} {'P':>7} {'R':>7} {'F1':>7} {'Thr':>7}")
for label in WEAK_LABELS:
    for model_name, report in [
        ("Champion", champion_weak_report),
        ("Pilot", pilot_weak_report),
    ]:
        r = report[label]
        print(
            f"{label:<12} {model_name:<10} {r['gold']:6d} "
            f"{r['raw_predictions']:6d} {r['exact_matches']:6d} "
            f"{r['precision']:7.3f} {r['recall']:7.3f} "
            f"{r['f1']:7.3f} {r['threshold']:7.3f}"
        )
        c = np.asarray(r["confidences"], dtype=float)
        if len(c):
            print(
                " " * 12,
                "confidence p10/median/p90:",
                np.round(np.quantile(c, [0.1, 0.5, 0.9]), 4).tolist()
            )
        else:
            print(" " * 12, "No raw proposals.")


In [ ]:

from collections import Counter
import numpy as np

raw_tag_counts = Counter()
viterbi_tag_counts = Counter()
weak_argmax_tokens = Counter()
weak_viterbi_tokens = Counter()

for doc_id, (offsets, logits) in doc_logits.items():
    # doc_logits was created by Cell 29; verify its structure.
    logits = np.asarray(logits)
    if logits.ndim != 2 or logits.shape[1] != len(BIO_LABELS):
        continue

    raw_ids = logits.argmax(axis=1).tolist()
    path_ids = bio_viterbi_decode(logits)

    raw_tag_counts.update(BIO_LABELS[i] for i in raw_ids)
    viterbi_tag_counts.update(BIO_LABELS[i] for i in path_ids)

    for tag in WEAK_LABELS:
        weak_argmax_tokens[tag] += sum(
            BIO_LABELS[i] in (f"B-{tag}", f"I-{tag}") for i in raw_ids
        )
        weak_viterbi_tokens[tag] += sum(
            BIO_LABELS[i] in (f"B-{tag}", f"I-{tag}") for i in path_ids
        )

print("Top raw argmax BIO tags:")
for tag, count in raw_tag_counts.most_common(15):
    print(f"{tag:<14} {count:>8}")

print("\nWeak-label token counts: raw argmax vs Viterbi")
for tag in WEAK_LABELS:
    print(
        f"{tag:<12} raw={weak_argmax_tokens[tag]:>6} "
        f"viterbi={weak_viterbi_tokens[tag]:>6}"
    )

print("\nTop Viterbi BIO tags:")
for tag, count in viterbi_tag_counts.most_common(15):
    print(f"{tag:<14} {count:>8}")


In [ ]:

import numpy as np
from collections import defaultdict

def spans_from_raw_argmax(text, offsets, logits):
    pred_ids = np.asarray(logits).argmax(axis=1).tolist()
    return decode_spans(text, offsets, pred_ids)

def spans_from_viterbi(text, offsets, logits):
    pred_ids = bio_viterbi_decode(logits)
    return decode_spans(text, offsets, pred_ids)

raw_argmax_predictions = {}
viterbi_predictions = {}

for doc_id, text in val_texts.items():
    offsets, logits = doc_logits[doc_id]

    raw_argmax_predictions[doc_id] = spans_from_raw_argmax(
        text, offsets, logits
    )
    viterbi_predictions[doc_id] = spans_from_viterbi(
        text, offsets, logits
    )

# Raw decoding has no confidence filtering; Viterbi has no confidence filtering.
raw_metrics = calculate_exact_span_metrics(
    val_gold, raw_argmax_predictions, ENTITY_LABELS
)
viterbi_metrics = calculate_exact_span_metrics(
    val_gold, viterbi_predictions, ENTITY_LABELS
)

print("Exact-span macro-F1, without confidence thresholds")
print(f"Raw argmax: {raw_metrics['MACRO_F1']:.6f}")
print(f"Viterbi:    {viterbi_metrics['MACRO_F1']:.6f}")
print("\nPer-label comparison:")
print(f"{'Label':<12} {'Raw argmax':>12} {'Viterbi':>12} {'Delta':>10}")

for label in ENTITY_LABELS:
    r = raw_metrics[label]["f1"]
    v = viterbi_metrics[label]["f1"]
    print(f"{label:<12} {r:12.4f} {v:12.4f} {v-r:+10.4f}")

print("\nWeak-label raw span counts:")
for label in WEAK_LABELS:
    raw_n = sum(
        1 for spans in raw_argmax_predictions.values()
        for s, e, lab in spans if lab == label
    )
    vit_n = sum(
        1 for spans in viterbi_predictions.values()
        for s, e, lab in spans if lab == label
    )
    print(f"{label:<12} raw={raw_n:4d}  viterbi={vit_n:4d}")


In [ ]:

import numpy as np
from collections import defaultdict

weak_bio_stats = defaultdict(lambda: {
    "B_probs": [], "I_probs": [], "B_ranks": [], "I_ranks": []
})

for doc_id, gold_anns in val_gold.items():
    offsets, logits = doc_logits[doc_id]
    logits = np.asarray(logits, dtype=np.float32)

    shifted = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    probs /= probs.sum(axis=1, keepdims=True)

    for ann in gold_anns:
        start, end, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in WEAK_LABELS:
            continue

        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]
        if not token_indices:
            continue

        for j, i in enumerate(token_indices):
            prefix = "B" if j == 0 else "I"
            correct_tag = f"{prefix}-{label}"
            correct_id = label2id[correct_tag]
            row = probs[i]

            # Rank 1 means the correct BIO class has the highest probability.
            rank = 1 + int(np.sum(row > row[correct_id]))
            weak_bio_stats[label][f"{prefix}_probs"].append(
                float(row[correct_id])
            )
            weak_bio_stats[label][f"{prefix}_ranks"].append(rank)

print(f"{'Label':<12} {'Pos':<4} {'N':>6} {'Mean P':>9} {'Median P':>10} {'Top-1 %':>9} {'Median rank':>12}")
for label in WEAK_LABELS:
    for prefix in ["B", "I"]:
        probs = np.asarray(weak_bio_stats[label][f"{prefix}_probs"])
        ranks = np.asarray(weak_bio_stats[label][f"{prefix}_ranks"])
        if len(probs) == 0:
            continue
        print(
            f"{label:<12} {prefix:<4} {len(probs):6d} "
            f"{probs.mean():9.4f} {np.median(probs):10.4f} "
            f"{100*np.mean(ranks == 1):8.2f}% {np.median(ranks):12.1f}"
        )


In [ ]:

import numpy as np
import torch
from collections import defaultdict
from tqdm.auto import tqdm

def gold_weak_token_probabilities(model, texts, gold):
    stats = defaultdict(lambda: {"probs": [], "top1": []})
    model.eval()

    with torch.inference_mode():
        for doc_id, text in tqdm(texts.items(), desc="Scoring gold tokens"):
            enc = tokenizer.encode(text, add_special_tokens=False)
            ids, offsets = enc.ids, enc.offsets
            if not ids:
                continue

            sums = torch.zeros((len(ids), len(BIO_LABELS)), dtype=torch.float32)
            counts = torch.zeros(len(ids), dtype=torch.float32)

            for start in range(0, len(ids), EVAL_STRIDE):
                end = min(start + EVAL_MAX_LEN, len(ids))
                x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
                mask = torch.ones_like(x)

                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(x, mask)[0].float()

                sums[start:end] += logits[0].cpu()
                counts[start:end] += 1
                if end == len(ids):
                    break

            avg = sums / counts.clamp_min(1).unsqueeze(-1)
            probs = torch.softmax(avg, dim=-1).numpy()
            pred = avg.argmax(dim=-1).numpy()

            for ann in gold.get(doc_id, []):
                s, e, label = int(ann[0]), int(ann[1]), ann[2]
                if label not in WEAK_LABELS:
                    continue

                token_ids = [
                    i for i, (a, b) in enumerate(offsets)
                    if b > a and a < e and b > s
                ]
                for j, i in enumerate(token_ids):
                    prefix = "B" if j == 0 else "I"
                    tag = f"{prefix}-{label}"
                    target = label2id[tag]
                    key = (label, prefix)
                    stats[key]["probs"].append(float(probs[i, target]))
                    stats[key]["top1"].append(int(pred[i] == target))

    return stats

champion_prob_stats = gold_weak_token_probabilities(
    champion_model, val_texts, val_gold
)
pilot_prob_stats = gold_weak_token_probabilities(
    targeted_model, val_texts, val_gold
)

print(f"{'Label':<12} {'Pos':<3} {'Champion P':>11} {'Pilot P':>9} {'Delta P':>9} {'Champ top1':>11} {'Pilot top1':>10}")
for label in WEAK_LABELS:
    for prefix in ["B", "I"]:
        key = (label, prefix)
        c = champion_prob_stats[key]
        p = pilot_prob_stats[key]
        if not c["probs"]:
            continue

        cp = float(np.mean(c["probs"]))
        pp = float(np.mean(p["probs"]))
        ca = 100 * float(np.mean(c["top1"]))
        pa = 100 * float(np.mean(p["top1"]))

        print(
            f"{label:<12} {prefix:<3} {cp:11.4f} {pp:9.4f} "
            f"{pp-cp:+9.4f} {ca:10.2f}% {pa:9.2f}%"
        )


In [ ]:

import numpy as np
import torch

doc_id = next(iter(val_texts))
text = val_texts[doc_id]
enc = tokenizer.encode(text, add_special_tokens=False)
ids = enc.ids

cached_offsets, cached_logits = doc_logits[doc_id]
cached_logits = np.asarray(cached_logits, dtype=np.float32)

fresh_sum = torch.zeros((len(ids), len(BIO_LABELS)), dtype=torch.float32)
fresh_count = torch.zeros(len(ids), dtype=torch.float32)

champion_model.eval()
with torch.inference_mode():
    for start in range(0, len(ids), EVAL_STRIDE):
        end = min(start + EVAL_MAX_LEN, len(ids))
        x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
        mask = torch.ones_like(x)

        # Output is [batch, sequence, labels]; select batch once.
        logits = champion_model(x, mask)[0].float()

        assert logits.shape == (end - start, len(BIO_LABELS))
        fresh_sum[start:end] += logits.cpu()
        fresh_count[start:end] += 1

        if end == len(ids):
            break

fresh_logits = (fresh_sum / fresh_count.clamp_min(1).unsqueeze(-1)).numpy()

print("Document:", doc_id)
print("Cached shape:", cached_logits.shape)
print("Fresh shape:", fresh_logits.shape)
print("Offsets identical:", cached_offsets == enc.offsets)

diff = np.abs(cached_logits - fresh_logits)
print("Maximum logit difference:", float(diff.max()))
print("Mean absolute logit difference:", float(diff.mean()))
print(
    "Cached vs fresh argmax agreement:",
    float(np.mean(cached_logits.argmax(1) == fresh_logits.argmax(1)))
)


In [ ]:

import numpy as np
from collections import defaultdict

corrected_stats = defaultdict(lambda: {"probs": [], "top1": []})

for doc_id, gold_anns in val_gold.items():
    offsets, logits = doc_logits[doc_id]
    logits = np.asarray(logits, dtype=np.float32)

    # Stable softmax across the 25 BIO labels.
    shifted = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    probs /= probs.sum(axis=1, keepdims=True)
    pred_ids = logits.argmax(axis=1)

    for ann in gold_anns:
        s, e, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in WEAK_LABELS:
            continue

        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]

        for j, i in enumerate(token_indices):
            prefix = "B" if j == 0 else "I"
            tag = f"{prefix}-{label}"
            target_id = label2id[tag]
            key = (label, prefix)

            corrected_stats[key]["probs"].append(
                float(probs[i, target_id])
            )
            corrected_stats[key]["top1"].append(
                int(pred_ids[i] == target_id)
            )

print(f"{'Label':<12} {'Pos':<3} {'N':>6} {'Mean P':>9} {'Median P':>10} {'Top-1 %':>9}")
for label in WEAK_LABELS:
    for prefix in ["B", "I"]:
        values = corrected_stats[(label, prefix)]
        if not values["probs"]:
            continue

        p = np.asarray(values["probs"])
        top1 = np.asarray(values["top1"])

        print(
            f"{label:<12} {prefix:<3} {len(p):6d} "
            f"{p.mean():9.4f} {np.median(p):10.4f} "
            f"{100*top1.mean():8.2f}%"
        )


In [ ]:

import os
import torch

assert not champion_model.training, "Champion should remain in eval mode."

champion_state = champion_model.state_dict()
max_diff = max(
    (champion_state[k].detach().cpu() - champion_reference_cpu[k]).abs().max().item()
    for k in champion_state
)

pilot_path = "/kaggle/working/phase2_ner_targeted_pilot/targeted_pilot_8_steps.pt"

print("Champion max parameter difference from reference:", max_diff)
print("Pilot checkpoint exists:", os.path.exists(pilot_path))
if os.path.exists(pilot_path):
    print("Pilot checkpoint size (MiB):", round(os.path.getsize(pilot_path) / 1024**2, 2))

assert max_diff == 0.0
assert os.path.exists(pilot_path)
print("Champion integrity check passed.")


In [ ]:

import os

paths_to_check = [
    "/kaggle/working/cs781_backup_download/cs781_recovery/champion_recovery.pt",
    "/kaggle/working/cs781_backup_download/cs781_recovery/champion_thresholds_recovery.json",
    "/kaggle/working/phase2_ner_targeted_pilot/targeted_pilot_8_steps.pt",
    "/kaggle/working/phase2_ner_weighted/best_weighted_model.pt",
    "/kaggle/working/phase2_ner_experiment_E/lowest_train_loss.pt",
    "/kaggle/working/phase2_ner_experiment_F/experiment_F_epoch_1.pt",
    "/kaggle/working/prediction_champion.json",
    "/kaggle/working/prediction.json",
]

for path in paths_to_check:
    if os.path.isfile(path):
        size_mib = os.path.getsize(path) / (1024 ** 2)
        print(f"FOUND  {size_mib:8.2f} MiB  {path}")
    else:
        print(f"MISSING              {path}")


In [ ]:

import torch
import os

pilot_path = "/kaggle/working/phase2_ner_targeted_pilot/targeted_pilot_8_steps.pt"

pilot_ckpt = torch.load(
    pilot_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", list(pilot_ckpt.keys()))
print("Optimizer steps:", pilot_ckpt.get("optimizer_steps"))
print("Training loss history:", pilot_ckpt.get("loss_history"))
print("BIO label count:", len(pilot_ckpt.get("bio_labels", [])))
print("Labels match champion:",
      pilot_ckpt.get("bio_labels") == BIO_LABELS)
print("Checkpoint size (MiB):",
      round(os.path.getsize(pilot_path) / 1024**2, 2))


In [ ]:

import inspect

functions_to_check = [
    "make_unique_loss_windows",
    "decode_spans",
    "predict_doc_constrained_with_confidence",
    "filter_predictions_by_thresholds_for_texts",
    "calculate_exact_span_metrics",
    "bio_viterbi_decode",
]

for name in functions_to_check:
    fn = globals().get(name)
    print(f"\n{'='*60}\n{name}")
    if not callable(fn):
        print("MISSING")
        continue

    try:
        print("Signature:", inspect.signature(fn))
        print(inspect.getsource(fn)[:3000])
    except (OSError, TypeError) as exc:
        print("Python source unavailable:", repr(exc))

print("\nOriginal notebook training window settings:")
print("MAX_LEN = 512")
print("WINDOW_STRIDE = 448")


In [ ]:
# Restore the original notebook's inference window settings.
MAX_LEN = 512
WINDOW_STRIDE = 448

EVAL_MAX_LEN = 512
EVAL_STRIDE = 448

assert MAX_LEN == 512
assert WINDOW_STRIDE == 448
assert EVAL_MAX_LEN == 512
assert EVAL_STRIDE == 448

print("Original window settings restored.")
print("MAX_LEN:", MAX_LEN)
print("WINDOW_STRIDE:", WINDOW_STRIDE)
print("Champion unchanged:", not champion_model.training)
print("Pilot is separate:", targeted_model is not champion_model)

In [ ]:
import torch

assert targeted_model is not champion_model

_original_ner_model = ner_model

try:
    # 1. Generate raw champion predictions for every validation document.
    ner_model = champion_model
    champion_model.eval()

    champion_raw = {}
    for doc_id, text in val_texts.items():
        champion_raw[doc_id] = predict_doc_constrained_with_confidence(text)

    print("Champion raw predictions:", len(champion_raw))

    # 2. Generate raw pilot predictions for every validation document.
    ner_model = targeted_model
    targeted_model.eval()

    pilot_raw = {}
    for doc_id, text in val_texts.items():
        pilot_raw[doc_id] = predict_doc_constrained_with_confidence(text)

    print("Pilot raw predictions:", len(pilot_raw))

finally:
    ner_model = _original_ner_model
    champion_model.eval()
    targeted_model.eval()

# 3. Apply thresholds using the function's actual argument order:
#    (predictions, thresholds, texts)
champion_predictions = filter_predictions_by_thresholds_for_texts(
    champion_raw, champion_thresholds_fusion, val_texts
)

pilot_predictions = filter_predictions_by_thresholds_for_texts(
    pilot_raw, champion_thresholds_fusion, val_texts
)

# 4. Evaluate exact-span metrics.
champion_metrics = calculate_exact_span_metrics(
    val_gold, champion_predictions, ENTITY_LABELS
)

pilot_metrics = calculate_exact_span_metrics(
    val_gold, pilot_predictions, ENTITY_LABELS
)

print("\nChampion Macro-F1:", champion_metrics["MACRO_F1"])
print("Pilot Macro-F1:   ", pilot_metrics["MACRO_F1"])
print(
    "Difference:       ",
    pilot_metrics["MACRO_F1"] - champion_metrics["MACRO_F1"]
)

print("\nPer-label comparison")
print(f"{'Label':<12} {'Champion F1':>12} {'Pilot F1':>12} {'Difference':>12}")

for label in ENTITY_LABELS:
    c_f1 = champion_metrics[label]["f1"]
    p_f1 = pilot_metrics[label]["f1"]
    print(f"{label:<12} {c_f1:>12.4f} {p_f1:>12.4f} {p_f1-c_f1:>12.4f}")

print("\nPredicted spans:",
      sum(len(v) for v in champion_predictions.values()), "(champion),",
      sum(len(v) for v in pilot_predictions.values()), "(pilot)")

In [ ]:
import torch
import numpy as np
from collections import Counter, defaultdict

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

# Use the champion explicitly; do not change its weights.
champion_model.eval()

weak_token_counts = {
    label: Counter() for label in WEAK_LABELS
}
weak_start_counts = {
    label: Counter() for label in WEAK_LABELS
}
weak_examples = defaultdict(list)

# Cache per-document token predictions and probabilities.
champion_model_gold_audit = {}

for doc_id, text in train_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    token_ids = enc.ids
    offsets = enc.offsets

    # Use the same original window settings as inference.
    window_logits = []
    window_positions = []

    for start in range(0, len(token_ids), WINDOW_STRIDE):
        end = min(start + MAX_LEN, len(token_ids))
        if start >= len(token_ids):
            break

        ids = torch.tensor(
            [token_ids[start:end]], dtype=torch.long, device=device
        )
        mask = torch.ones_like(ids)

        with torch.no_grad():
            logits = champion_model(ids, mask)[0].float().cpu().numpy()

        window_logits.append((start, end, logits[0]))

        if end == len(token_ids):
            break

    # Average overlapping-window logits for each original token.
    sums = np.zeros((len(token_ids), len(BIO_LABELS)), dtype=np.float64)
    counts = np.zeros(len(token_ids), dtype=np.float64)

    for start, end, logits in window_logits:
        sums[start:end] += logits
        counts[start:end] += 1

    counts[counts == 0] = 1
    avg_logits = sums / counts[:, None]

    # Softmax probabilities for token-level diagnostics.
    shifted = avg_logits - avg_logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    pred_ids = avg_logits.argmax(axis=1)

    champion_model_gold_audit[doc_id] = {
        "offsets": offsets,
        "pred_ids": pred_ids,
        "probs": probs,
    }

    # Gold format: [start, end, label, surface]
    for gold in train_gold[doc_id]:
        gs, ge, label = int(gold[0]), int(gold[1]), gold[2]
        if label not in WEAK_LABELS:
            continue

        token_indices = [
            i for i, (s, e) in enumerate(offsets)
            if e > s and s < ge and e > gs
        ]
        if not token_indices:
            continue

        for j, i in enumerate(token_indices):
            predicted_tag = BIO_LABELS[int(pred_ids[i])]
            weak_token_counts[label][predicted_tag] += 1

            if j == 0:
                weak_start_counts[label][predicted_tag] += 1

        # Save a few examples for manual inspection.
        if len(weak_examples[label]) < 8:
            weak_examples[label].append({
                "doc_id": doc_id,
                "gold_text": text[gs:ge],
                "gold_span": (gs, ge),
                "predicted_tags": [
                    BIO_LABELS[int(pred_ids[i])] for i in token_indices
                ],
                "top_predictions": [
                    sorted(
                        [
                            (BIO_LABELS[k], float(probs[i, k]))
                            for k in range(len(BIO_LABELS))
                        ],
                        key=lambda x: x[1],
                        reverse=True
                    )[:3]
                    for i in token_indices[:5]
                ],
            })

print("Token predictions over gold weak-label spans")
for label in WEAK_LABELS:
    total = sum(weak_token_counts[label].values())
    print(f"\n{label}: {total} tokens")
    for tag, count in weak_token_counts[label].most_common(10):
        print(f"  {tag:<14} {count:>6} ({count / max(total, 1):.1%})")

print("\nPredictions at the first token of each gold span")
for label in WEAK_LABELS:
    total = sum(weak_start_counts[label].values())
    print(f"\n{label}: {total} entity starts")
    for tag, count in weak_start_counts[label].most_common(8):
        print(f"  {tag:<14} {count:>5} ({count / max(total, 1):.1%})")

In [ ]:
for label in WEAK_LABELS:
    print("\n" + "=" * 90)
    print("LABEL:", label)

    for ex in weak_examples[label]:
        print(f"\nDocument: {ex['doc_id']}")
        print("Gold text:", repr(ex["gold_text"]))
        print("Gold offsets:", ex["gold_span"])
        print("Predicted BIO tags:", ex["predicted_tags"])

        for i, top3 in enumerate(ex["top_predictions"]):
            formatted = ", ".join(
                f"{tag}={prob:.3f}" for tag, prob in top3
            )
            print(f"  Token {i + 1}: {formatted}")

In [ ]:
val_weak_token_counts = {
    label: Counter() for label in WEAK_LABELS
}

for doc_id, text in val_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    token_ids = enc.ids
    offsets = enc.offsets

    sums = np.zeros((len(token_ids), len(BIO_LABELS)), dtype=np.float64)
    counts = np.zeros(len(token_ids), dtype=np.float64)

    for start in range(0, len(token_ids), WINDOW_STRIDE):
        end = min(start + MAX_LEN, len(token_ids))
        if start >= len(token_ids):
            break

        ids = torch.tensor(
            [token_ids[start:end]], dtype=torch.long, device=device
        )
        mask = torch.ones_like(ids)

        with torch.no_grad():
            logits = champion_model(ids, mask)[0].float().cpu().numpy()[0]

        sums[start:end] += logits
        counts[start:end] += 1

        if end == len(token_ids):
            break

    counts[counts == 0] = 1
    pred_ids = (sums / counts[:, None]).argmax(axis=1)

    for gold in val_gold[doc_id]:
        gs, ge, label = int(gold[0]), int(gold[1]), gold[2]
        if label not in WEAK_LABELS:
            continue

        for i, (s, e) in enumerate(offsets):
            if e > s and s < ge and e > gs:
                val_weak_token_counts[label][BIO_LABELS[int(pred_ids[i])]] += 1

print("Validation token predictions over gold weak-label spans")
for label in WEAK_LABELS:
    counts = val_weak_token_counts[label]
    total = sum(counts.values())
    print(f"\n{label}: {total} tokens")
    for tag, count in counts.most_common(8):
        print(f"  {tag:<14} {count:>6} ({count / max(total, 1):.1%})")

In [ ]:
from collections import Counter

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

audit = {
    label: {
        "gold_tokens": 0,
        "matching_bio": 0,
        "wrong_bio": 0,
        "examples": [],
    }
    for label in WEAK_LABELS
}

for doc_id, text in train_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    offsets = enc.offsets

    for ann in train_gold[doc_id]:
        gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in WEAK_LABELS:
            continue

        indices = [
            i for i, (s, e) in enumerate(offsets)
            if e > s and s < ge and e > gs
        ]

        for j, i in enumerate(indices):
            expected = ("B-" if j == 0 else "I-") + label
            audit[label]["gold_tokens"] += 1

            if expected in BIO_LABELS:
                audit[label]["matching_bio"] += 1
            else:
                audit[label]["wrong_bio"] += 1

        if len(audit[label]["examples"]) < 3:
            audit[label]["examples"].append({
                "text": text[gs:ge],
                "expected_tags": [
                    ("B-" if j == 0 else "I-") + label
                    for j in range(len(indices))
                ],
                "token_offsets": [offsets[i] for i in indices],
            })

for label in WEAK_LABELS:
    a = audit[label]
    print(
        f"{label}: tokens={a['gold_tokens']}, "
        f"BIO labels available={a['matching_bio']}, "
        f"unavailable={a['wrong_bio']}"
    )
    for ex in a["examples"]:
        print(" ", repr(ex["text"]))
        print("   expected:", ex["expected_tags"])
        print("   offsets: ", ex["token_offsets"])

In [ ]:
import inspect
from collections import Counter

print("Checking available window-building functions...")

for name in [
    "document_to_windows",
    "make_unique_loss_windows",
    "build_training_windows",
]:
    obj = globals().get(name)
    if callable(obj):
        print(f"\n{name} exists")
        try:
            print("Signature:", inspect.signature(obj))
            print("Source preview:")
            print(inspect.getsource(obj)[:1200])
        except Exception as e:
            print("Source unavailable:", repr(e))
    else:
        print(f"{name}: not defined")

print("\nChecking existing training-window variables...")
for name, obj in list(globals().items()):
    if any(term in name.lower() for term in ["window", "train_dataset", "dataset"]):
        if isinstance(obj, (list, tuple)):
            print(f"{name}: {type(obj).__name__}, length={len(obj)}")
        elif obj is not None and any(
            term in type(obj).__name__.lower()
            for term in ["dataset", "dataloader"]
        ):
            print(f"{name}: {type(obj).__name__}")

In [ ]:
from collections import Counter

print("Number of pilot windows:", len(pilot_windows))
print("First window type:", type(pilot_windows[0]).__name__)

w = pilot_windows[0]

if isinstance(w, dict):
    print("Window keys:", list(w.keys()))

    for key, value in w.items():
        try:
            print(
                f"{key}: type={type(value).__name__}, "
                f"length={len(value)}, "
                f"sample={str(value[:8])[:250]}"
            )
        except Exception:
            print(f"{key}: type={type(value).__name__}, value={str(value)[:200]}")

elif isinstance(w, (list, tuple)):
    print("Number of fields:", len(w))
    for i, value in enumerate(w):
        try:
            print(
                f"field {i}: type={type(value).__name__}, "
                f"length={len(value)}, "
                f"sample={str(value[:8])[:250]}"
            )
        except Exception:
            print(f"field {i}: {type(value).__name__}, {str(value)[:200]}")

print("\nExisting relevant globals:")
for name in [
    "BIO_LABELS", "label2id", "train_texts", "train_gold",
    "tokenizer", "MAX_LEN", "WINDOW_STRIDE",
]:
    print(name, "exists" if name in globals() else "MISSING")

In [ ]:
from collections import Counter, defaultdict

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

stored_counts = Counter()
stored_b_counts = Counter()
stored_i_counts = Counter()
ignored_counts = Counter()
examples = defaultdict(list)

for w in pilot_windows:
    labels = w["labels"]

    for i, label_id in enumerate(labels):
        label_id = int(label_id)

        if label_id == -100:
            ignored_counts["all"] += 1
            continue

        if not (0 <= label_id < len(BIO_LABELS)):
            raise ValueError(f"Invalid label ID: {label_id}")

        tag = BIO_LABELS[label_id]

        for entity in WEAK_LABELS:
            if tag in (f"B-{entity}", f"I-{entity}"):
                stored_counts[entity] += 1

                if tag.startswith("B-"):
                    stored_b_counts[entity] += 1
                else:
                    stored_i_counts[entity] += 1

                if len(examples[entity]) < 3:
                    examples[entity].append({
                        "doc_id": w["doc_id"],
                        "window_start": w["start"],
                        "token_position": i,
                        "tag": tag,
                        "token_id": w["input_ids"][i],
                    })

print("Stored weak-label tags in pilot windows")
for entity in WEAK_LABELS:
    print(
        f"{entity:<12} B={stored_b_counts[entity]:>5} "
        f"I={stored_i_counts[entity]:>5} "
        f"total={stored_counts[entity]:>5}"
    )
    for ex in examples[entity]:
        print("  ", ex)

print("\nIgnored label positions:", ignored_counts["all"])
print("\nExpected source BIO token counts:")
for entity in WEAK_LABELS:
    expected_b = sum(
        1 for doc_id in train_gold
        for ann in train_gold[doc_id]
        if ann[2] == entity
    )
    print(f"{entity:<12} gold entity spans={expected_b}")

In [ ]:
import torch
import numpy as np
from collections import defaultdict

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

def collect_gold_weak_probabilities(model, texts, gold_data):
    model.eval()
    result = {
        label: {"B": [], "I": []}
        for label in WEAK_LABELS
    }

    for doc_id, text in texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)
        token_ids, offsets = enc.ids, enc.offsets

        # Match the original inference settings.
        sums = np.zeros((len(token_ids), len(BIO_LABELS)), dtype=np.float64)
        counts = np.zeros(len(token_ids), dtype=np.float64)

        for start in range(0, len(token_ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(token_ids))
            if start >= len(token_ids):
                break

            ids = torch.tensor(
                [token_ids[start:end]], dtype=torch.long, device=device
            )
            mask = torch.ones_like(ids)

            with torch.no_grad():
                logits = model(ids, mask)[0].float().cpu().numpy()[0]

            sums[start:end] += logits
            counts[start:end] += 1

            if end == len(token_ids):
                break

        counts[counts == 0] = 1
        avg_logits = sums / counts[:, None]
        shifted = avg_logits - avg_logits.max(axis=1, keepdims=True)
        exp_logits = np.exp(shifted)
        probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)

        for ann in gold_data[doc_id]:
            gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
            if label not in WEAK_LABELS:
                continue

            indices = [
                i for i, (s, e) in enumerate(offsets)
                if e > s and s < ge and e > gs
            ]

            for j, i in enumerate(indices):
                role = "B" if j == 0 else "I"
                tag = f"{role}-{label}"
                result[label][role].append(
                    float(probs[i, label2id[tag]])
                )

    return result

champion_probs = collect_gold_weak_probabilities(
    champion_model, train_texts, train_gold
)
pilot_probs = collect_gold_weak_probabilities(
    targeted_model, train_texts, train_gold
)

print("Mean probability assigned to the correct BIO class")
print(f"{'Label':<12} {'Role':<5} {'Champion':>12} {'Pilot':>12} {'Change':>12}")

for label in WEAK_LABELS:
    for role in ["B", "I"]:
        c = np.mean(champion_probs[label][role])
        p = np.mean(pilot_probs[label][role])
        print(f"{label:<12} {role:<5} {c:>12.5f} {p:>12.5f} {p-c:>+12.5f}")

In [ ]:
import numpy as np, pandas as pd, torch

DIAG_MAX_LEN, DIAG_STRIDE = 512, 448          # original inference settings
DIAG_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT", "COURT", "DATE"]  # last two = reference
O_ID = label2id["O"]

@torch.inference_mode()
def diag_doc_probs(model, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    dev = next(model.parameters()).device
    n = len(ids)
    logit_sum = torch.zeros((n, len(BIO_LABELS)))
    cnt = torch.zeros(n)
    model.eval()
    for start in range(0, n, DIAG_STRIDE):
        end = min(start + DIAG_MAX_LEN, n)
        x = torch.tensor([ids[start:end]], dtype=torch.long, device=dev)
        out = model(x, torch.ones_like(x))
        out = out[0] if isinstance(out, tuple) else out
        logit_sum[start:end] += out[0].float().cpu()
        cnt[start:end] += 1
        if end == n:
            break
    avg = logit_sum / cnt.clamp_min(1).unsqueeze(-1)
    return offsets, torch.softmax(avg, dim=-1)

def gold_token_table(model, name):
    rows = {lab: [] for lab in DIAG_LABELS}   # (is_first, p_correct_tag, p_O, p_type_any, argmax_id)
    for doc_id, text in val_texts.items():
        offsets, probs = diag_doc_probs(model, text)
        for ent in val_gold[doc_id]:
            s, e, lab = int(ent[0]), int(ent[1]), ent[2]
            if lab not in rows:
                continue
            toks = [i for i, (a, b) in enumerate(offsets) if b > a and a < e and b > s]
            for k, i in enumerate(toks):
                is_first = (k == 0)
                tag = f"{'B' if is_first else 'I'}-{lab}"
                p = probs[i]
                p_type = float(p[label2id[f"B-{lab}"]] + p[label2id[f"I-{lab}"]])
                rows[lab].append((is_first, float(p[label2id[tag]]), float(p[O_ID]),
                                  p_type, int(p.argmax())))
    out = []
    for lab, r in rows.items():
        a = np.array(r, dtype=float)
        for grp, mask in [("B (first tok)", a[:, 0] == 1), ("I (rest)", a[:, 0] == 0)]:
            m = a[mask]
            out.append({
                "model": name, "label": lab, "tok": grp, "n": len(m),
                "mean_p_correct": m[:, 1].mean(), "median_p_correct": np.median(m[:, 1]),
                "frac_p>0.10": (m[:, 1] > 0.10).mean(),
                "frac_p>0.30": (m[:, 1] > 0.30).mean(),
                "mean_p_O": m[:, 2].mean(),
                "mean_p_B+I_of_type": m[:, 3].mean(),
                "frac_argmax_O": (m[:, 4] == O_ID).mean(),
            })
    return pd.DataFrame(out)

champ_tbl = gold_token_table(champion_model, "champion")
pilot_tbl = gold_token_table(targeted_model, "pilot")
diag = pd.concat([champ_tbl, pilot_tbl]).sort_values(["label", "tok", "model"])
pd.set_option("display.width", 220)
print(diag.round(3).to_string(index=False))

In [ ]:
ENT = ["APP","RESP","A.COUNSEL","R.COUNSEL","JUDGE","WIT","AUTH","COURT","STAT","PREC","DATE","CASENO"]
TYPES = ["O"] + ENT
type_ids = [TYPES.index("O" if t == "O" else t[2:]) for t in BIO_LABELS]
M = torch.zeros(len(BIO_LABELS), len(TYPES))
M[torch.arange(len(BIO_LABELS)), torch.tensor(type_ids)] = 1.0
type_of_id = torch.tensor(type_ids)
WEAK = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

def type_confusion(model, texts, gold, fold_name):
    mass = {l: [] for l in WEAK}
    arg = {l: [] for l in WEAK}
    for doc_id, text in texts.items():
        offsets, probs = diag_doc_probs(model, text)
        t_mass = probs @ M                          # [n_tokens, 13] mass per entity type
        t_arg = type_of_id[probs.argmax(-1)]        # argmax type per token
        for ent in gold[doc_id]:
            lab = ent[2]
            if lab not in mass:
                continue
            s, e = int(ent[0]), int(ent[1])
            toks = [i for i, (a, b) in enumerate(offsets) if b > a and a < e and b > s]
            if toks:
                mass[lab].append(t_mass[toks])
                arg[lab].append(t_arg[toks])
    print(f"\n===== {fold_name} (champion) =====")
    for lab in WEAK:
        m = torch.cat(mass[lab]); a = torch.cat(arg[lab])
        mm = m.mean(0)
        top_mass = sorted(zip(TYPES, mm.tolist()), key=lambda x: -x[1])[:5]
        counts = torch.bincount(a, minlength=len(TYPES)).float() / len(a)
        top_arg = sorted(zip(TYPES, counts.tolist()), key=lambda x: -x[1])[:5]
        print(f"\n{lab}  (n_tokens={len(a)})")
        print("  mean prob mass by type :", "  ".join(f"{t}={v:.2f}" for t, v in top_mass))
        print("  argmax by type         :", "  ".join(f"{t}={v:.2f}" for t, v in top_arg))

type_confusion(champion_model, val_texts,   val_gold,   "FOLD 2 (validation)")
type_confusion(champion_model, train_texts, train_gold, "FOLD 1 (training)")

In [ ]:
from collections import Counter
import random
random.seed(0)

vit_counts = {l: Counter() for l in WEAK}
samples = {l: [] for l in WEAK}

for doc_id, text in val_texts.items():
    offsets, probs = diag_doc_probs(champion_model, text)
    vpath = bio_viterbi_decode(torch.log(probs.clamp_min(1e-12)))  # same path as on raw logits
    apath = probs.argmax(-1).tolist()
    for ent in val_gold[doc_id]:
        lab = ent[2]
        if lab not in vit_counts:
            continue
        s, e = int(ent[0]), int(ent[1])
        toks = [i for i, (a, b) in enumerate(offsets) if b > a and a < e and b > s]
        for i in toks:
            vit_counts[lab][TYPES[type_ids[vpath[i]]]] += 1
        if toks:
            samples[lab].append((
                text[max(0, s - 40):s].replace("\n", " "),
                text[s:e].replace("\n", " "),
                [BIO_LABELS[apath[i]] for i in toks[:8]],
                [BIO_LABELS[vpath[i]] for i in toks[:8]],
            ))

print("=== Viterbi-decoded type distribution on gold weak-label tokens (fold 2) ===")
for lab in WEAK:
    tot = sum(vit_counts[lab].values())
    top = vit_counts[lab].most_common(4)
    print(f"{lab:10s} n={tot}  " + "  ".join(f"{t}={c/tot:.2f}" for t, c in top))

print("\n=== Sample gold spans: left context | gold text | argmax tags | viterbi tags ===")
for lab in WEAK:
    print(f"\n--- {lab} ---")
    for ctx, gtxt, am, vt in random.sample(samples[lab], min(4, len(samples[lab]))):
        print(f"ctx : ...{ctx!r}")
        print(f"gold: {gtxt[:110]!r}")
        print(f"arg : {am}")
        print(f"vit : {vt}\n")

In [ ]:
import re
from collections import Counter

HON = re.compile(r"\b(?:Mr|Ms|Mrs|Shri|Sri|Smt|Dr)\b\.?[ \t]*(?=[A-Z])")

def covering_labels(gold_list, pos):
    return tuple(sorted({e[2] for e in gold_list if int(e[0]) <= pos < int(e[1])})) or ("NONE",)

def honorific_audit(texts, gold, fold_name):
    cnt, none_examples = Counter(), []
    for doc_id, text in texts.items():
        for m in HON.finditer(text):
            labs = covering_labels(gold[doc_id], m.end())
            cnt[labs] += 1
            if labs == ("NONE",) and len(none_examples) < 6:
                none_examples.append(text[max(0, m.start()-30):m.end()+25].replace("\n", " "))
    tot = sum(cnt.values())
    print(f"\n=== {fold_name}: honorific-prefixed names, covering gold label (total={tot}) ===")
    for labs, c in cnt.most_common(10):
        print(f"  {'+'.join(labs):28s} {c:5d}  {c/tot:5.1%}")
    print("  sample unannotated:")
    for ex in none_examples:
        print("   ", repr(ex))

honorific_audit(train_texts, train_gold, "FOLD 1 (train)")
honorific_audit(val_texts,   val_gold,   "FOLD 2 (val)")

print("\n=== Preceding word before gold weak-label spans (both folds) ===")
prev = {l: Counter() for l in WEAK}
for texts, gold in [(train_texts, train_gold), (val_texts, val_gold)]:
    for doc_id, text in texts.items():
        for e in gold[doc_id]:
            if e[2] in prev:
                w = text[max(0, int(e[0]) - 20):int(e[0])].split()
                prev[e[2]][(w[-1].lower() if w else "<bos>")] += 1
for lab in WEAK:
    tot = sum(prev[lab].values())
    print(f"{lab:10s} n={tot}  " + "  ".join(f"{w!r}={c}" for w, c in prev[lab].most_common(7)))

In [ ]:
import math, torch
from bisect import bisect_left, bisect_right

def audit_doc_windows(text, entities, max_len=512, stride=448):
    enc = tokenizer.encode(text, add_special_tokens=False)
    n, offsets = len(enc.ids), enc.offsets
    starts = [a for a, b in offsets]; ends = [b for a, b in offsets]
    tags = [label2id["O"]] * n
    for ent in entities:
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        lo, hi = bisect_right(ends, s), bisect_left(starts, e)       # tokens with end>s and start<e
        ov = [i for i in range(lo, hi) if ends[i] > starts[i]]
        if not ov:
            continue
        tags[ov[0]] = label2id[f"B-{lab}"]
        for i in ov[1:]:
            tags[i] = label2id[f"I-{lab}"]
    wins = []
    for st in range(0, n, stride):
        en = min(st + max_len, n)
        if st >= en:
            break
        w = tags[st:en]
        if w and BIO_LABELS[w[0]].startswith("I-"):
            w = [label2id["B-" + BIO_LABELS[w[0]][2:]]] + w[1:]
        wins.append(w)
        if en == n:
            break
    return wins, tags

doc_counts = torch.zeros(len(BIO_LABELS)); win_counts = torch.zeros(len(BIO_LABELS)); n_wins = 0
for doc_id, text in train_texts.items():
    wins, tags = audit_doc_windows(text, train_gold[doc_id])
    doc_counts += torch.bincount(torch.tensor(tags), minlength=len(BIO_LABELS)).float()
    for w in wins:
        win_counts += torch.bincount(torch.tensor(w), minlength=len(BIO_LABELS)).float()
    n_wins += len(wins)

print("Sanity check vs earlier audit (doc-level B/I):")
for lab, (b, i) in {"RESP": (191, 370), "A.COUNSEL": (70, 239), "R.COUNSEL": (87, 264), "WIT": (67, 130)}.items():
    print(f"  {lab:10s} mine B={int(doc_counts[label2id['B-'+lab]])} I={int(doc_counts[label2id['I-'+lab]])}   expected B={b} I={i}")

freq = win_counts.clamp_min(1)
raw = torch.sqrt(freq[0] / freq)
w = raw.clone(); w[0] = 1.0
w = w.clamp(1.0, 5.0)
w_norm = w / w.mean()
print(f"\nTrain windows: {n_wins}  ->  optimizer steps/epoch ~{math.ceil(n_wins/8)}, x3 epochs ~{3*math.ceil(n_wins/8)}")
print(f"\n{'tag':<14}{'tokens':>9}{'raw_w':>8}{'clamped':>9}{'final':>8}")
for i, t in enumerate(BIO_LABELS):
    print(f"{t:<14}{int(win_counts[i]):>9}{raw[i].item():>8.1f}{w[i].item():>9.1f}{w_norm[i].item():>8.2f}")
print("\nEntity tags at the clamp (5.0):", sum(1 for i in range(1, len(BIO_LABELS)) if w[i] >= 5.0), "of", len(BIO_LABELS) - 1)

In [ ]:

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Config, GPT2Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Part 0: locate and verify the champion checkpoint ----
def find_artifact(filename):
    matches = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if root.exists():
            matches.extend(
                p for p in root.rglob(filename) if p.is_file()
            )
    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename}")
    preferred = [p for p in matches if "cs781_recovery" in str(p)]
    return preferred[0] if preferred else matches[0]

RECOVERY_PATH = find_artifact("champion_recovery.pt")
PHASE1_PATH = find_artifact("checkpoint_latest.pt")

rec = torch.load(
    RECOVERY_PATH, map_location="cpu", weights_only=False
)
sd = champion_model.state_dict()

missing = set(rec["model_state_dict"]) - set(sd)
assert not missing, f"Champion model is missing keys: {sorted(missing)[:10]}"

maxdiff = max(
    (sd[k].detach().float().cpu() - v.float()).abs().max().item()
    for k, v in rec["model_state_dict"].items()
)
print("Champion vs recovery checkpoint max |diff|:", maxdiff)

if "targeted_model" in globals():
    print(
        "Champion encoder shares targeted_model encoder:",
        champion_model.encoder is targeted_model.encoder
    )
else:
    print("targeted_model not defined; skipping encoder-sharing check.")

del rec, sd
gc.collect()

# ---- Part 1: pristine frozen Phase 1 encoder ----
phase1_ckpt = torch.load(
    PHASE1_PATH, map_location="cpu", weights_only=False
)
cfg = phase1_ckpt["model_config"]

p1_cfg = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

enc_p1 = GPT2Model(p1_cfg)
enc_p1.load_state_dict(
    {
        k.removeprefix("transformer."): v
        for k, v in phase1_ckpt["model_state_dict"].items()
        if k.startswith("transformer.")
    },
    strict=True,
)
enc_p1 = enc_p1.to(device).eval()

# Estimate feature memory before extracting all token representations.
def token_count(text):
    return len(tokenizer.encode(text, add_special_tokens=False).ids)

n_train_tokens = sum(token_count(t) for t in train_texts.values())
n_val_tokens = sum(token_count(t) for t in val_texts.values())
estimated_gib = (
    (n_train_tokens + n_val_tokens) * p1_cfg.n_embd * 4
    / (1024 ** 3)
)
print("Train tokens:", n_train_tokens)
print("Validation tokens:", n_val_tokens)
print(f"Estimated float32 feature storage: {estimated_gib:.2f} GiB")

if estimated_gib > 8:
    raise MemoryError(
        "Feature tensors may exceed available RAM. Stop here and use a "
        "chunked/disk-backed feature extraction variant instead."
    )

@torch.inference_mode()
def doc_features(encoder, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids = enc.ids
    n = len(ids)
    feat = torch.zeros((n, p1_cfg.n_embd), dtype=torch.float32)
    cnt = torch.zeros(n, dtype=torch.float32)

    for st in range(0, n, 448):
        en = min(st + 512, n)
        x = torch.tensor([ids[st:en]], dtype=torch.long, device=device)
        h = encoder(
            input_ids=x,
            attention_mask=torch.ones_like(x)
        ).last_hidden_state[0].float().cpu()

        feat[st:en] += h
        cnt[st:en] += 1
        if en == n:
            break

    return feat / cnt.clamp_min(1).unsqueeze(-1)

def build(texts, gold, want_champion=False):
    X, Y, C = [], [], []
    for doc_id, text in texts.items():
        _, tags = audit_doc_windows(text, gold[str(doc_id)] if str(doc_id) in gold else gold[doc_id])
        X.append(doc_features(enc_p1, text))
        Y.append(torch.tensor(tags, dtype=torch.long))

        if want_champion:
            _, probs = diag_doc_probs(champion_model, text)
            C.append(probs.argmax(-1).cpu())

    X = torch.cat(X)
    Y = torch.cat(Y)
    C = torch.cat(C) if want_champion else None
    assert len(X) == len(Y), "Feature and BIO tag lengths do not match."
    if C is not None:
        assert len(C) == len(Y), "Champion prediction and gold lengths do not match."
    return X, Y, C

Xtr, Ytr, _ = build(train_texts, train_gold)
Xva, Yva, Cva = build(val_texts, val_gold, want_champion=True)
print("Feature shapes:", tuple(Xtr.shape), tuple(Xva.shape))

# ---- Part 2: train the linear probe ----
Xtr_g = Xtr.to(device)
Ytr_g = Ytr.to(device)

mu = Xtr_g.mean(0)
sdv = Xtr_g.std(0) + 1e-6

counts = torch.bincount(
    Ytr, minlength=len(BIO_LABELS)
).float().clamp_min(1)

w = torch.sqrt(counts[0] / counts)
w[0] = 1.0

head = nn.Linear(p1_cfg.n_embd, len(BIO_LABELS)).to(device)
opt = torch.optim.AdamW(
    head.parameters(), lr=1e-2, weight_decay=1e-4
)

for step in range(1, 401):
    logits = head((Xtr_g - mu) / sdv)
    loss = F.cross_entropy(logits, Ytr_g, weight=w.to(device))
    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % 100 == 0:
        print(f"step {step} loss {loss.item():.4f}")

with torch.no_grad():
    P_tr = head((Xtr_g - mu) / sdv).argmax(-1).cpu()
    P_va = head((Xva.to(device) - mu) / sdv).argmax(-1).cpu()

# ---- Part 3: token-level type P/R/F1 (B/I merged) ----
def prf(gold_ids, pred_ids):
    g_t = type_of_id[gold_ids]
    p_t = type_of_id[pred_ids]
    out = {}

    for t in ENT:
        j = TYPES.index(t)
        g = g_t == j
        p = p_t == j
        tp = (g & p).sum().item()
        P = tp / max(p.sum().item(), 1)
        R = tp / max(g.sum().item(), 1)
        F1 = 2 * P * R / (P + R) if P + R else 0.0
        out[t] = (P, R, F1)

    return out

probe_va = prf(Yva, P_va)
champ_va = prf(Yva, Cva)
probe_tr = prf(Ytr, P_tr)

rows = [{
    "label": t,
    "probe_val_P": probe_va[t][0],
    "probe_val_R": probe_va[t][1],
    "probe_val_F1": probe_va[t][2],
    "champ_val_P": champ_va[t][0],
    "champ_val_R": champ_va[t][1],
    "champ_val_F1": champ_va[t][2],
    "probe_train_F1": probe_tr[t][2],
} for t in ENT]

pd.set_option("display.width", 220)
print(pd.DataFrame(rows).round(3).to_string(index=False))

del Xtr_g, Ytr_g, Xtr, Xva
gc.collect()
torch.cuda.empty_cache()
print("\nLinear-probe experiment complete; no prediction files were written.")


In [ ]:

# FIX — Define the champion counsel baseline before calculating gain

if "c_f_thr" not in globals():
    if "champion_model" not in globals() or "val_texts" not in globals():
        raise NameError(
            "Need champion_model and val_texts. Run the champion-loading "
            "and validation setup cells first."
        )

    if "diag_doc_probs" not in globals():
        raise NameError(
            "diag_doc_probs is not defined. Run the original inference-helper "
            "cell before calculating the champion baseline."
        )

    # Use the notebook's original inference helper to recover champion
    # token predictions, then calculate the counsel baseline with its
    # existing evaluation function.
    print(
        "c_f_thr is missing. Do not calculate gain yet: first run the "
        "cell that evaluates champion predictions and defines c_f_thr."
    )
else:
    gain = (
        r_va["A.COUNSEL"] + r_va["R.COUNSEL"]
        - c_f_thr["A.COUNSEL"] - c_f_thr["R.COUNSEL"]
    ) / 12

    print(
        f"Macro-F1 delta on validation if counsel predictions are replaced: "
        f"{gain:+.4f}"
    )


In [ ]:

# FIX — Correct counsel-rule comparison
r_va = globals()["r_va"]

assert isinstance(r_va, dict), "r_va should contain per-label rule F1 scores."
assert all(k in r_va for k in ("A.COUNSEL", "R.COUNSEL")), (
    f"Unexpected r_va labels: {list(r_va.keys())}"
)

print("Fold 2 counsel-rule results:")
for lab in ("A.COUNSEL", "R.COUNSEL"):
    print(f"  {lab}: F1={r_va[lab]:.4f}")

print(f"\nMean F1 across the two counsel labels: "
      f"{(r_va['A.COUNSEL'] + r_va['R.COUNSEL']) / 2:.4f}")

print("\nChampion overall macro-F1 currently stored in c_f1:", c_f1)
print(
    "These values are not directly comparable. To calculate the true gain, "
    "we must evaluate the champion per-label on fold 2 with the same "
    "evaluation function used for the rules."
)


In [ ]:

# Inspect the notebook's evaluation helpers to find the correct baseline method
import inspect

for name in ("eval_rule", "score", "as_scored", "merge_policy",
             "predict_texts", "diag_doc_probs"):
    obj = globals().get(name)
    print(f"\n{name}: {type(obj).__name__ if obj is not None else 'MISSING'}")
    if callable(obj):
        try:
            print(inspect.signature(obj))
            print(inspect.getsource(obj)[:1800])
        except Exception as e:
            print("Source unavailable:", e)


In [ ]:

# CELL — Exact-span counsel baseline for the champion on fold 2
from collections import defaultdict, Counter
import torch

assert "champion_model" in globals()
assert "val_texts" in globals() and "val_gold" in globals()
assert "tokenizer" in globals() and "BIO_LABELS" in globals()
assert "prf_sets" in globals(), "Missing original prf_sets evaluation helper."

# Gold spans for the two counsel labels.
G = defaultdict(set)
for d, ents in val_gold.items():
    for e in ents:
        if len(e) >= 3 and e[2] in ("A.COUNSEL", "R.COUNSEL"):
            G[e[2]].add((d, int(e[0]), int(e[1])))

P = defaultdict(set)
id_to_bio = {i: lab for i, lab in enumerate(BIO_LABELS)}

def decode_bio_spans(text, offsets, pred_ids):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    for (start, end), idx in zip(offsets, pred_ids):
        bio = id_to_bio[int(idx)]
        if start == end:
            continue

        if bio == "O":
            if current_label is not None:
                spans.append((current_start, current_end, current_label))
                current_label = None
            continue

        if "-" not in bio:
            continue

        prefix, label = bio.split("-", 1)
        if prefix == "B" or label != current_label:
            if current_label is not None:
                spans.append((current_start, current_end, current_label))
            current_label = label
            current_start, current_end = start, end
        elif prefix == "I" and current_label == label:
            current_end = end

    if current_label is not None:
        spans.append((current_start, current_end, current_label))

    return spans

for d, text in val_texts.items():
    offsets, probs = diag_doc_probs(champion_model, text)
    pred_ids = probs.argmax(-1).tolist()

    for s, e, lab in decode_bio_spans(text, offsets, pred_ids):
        if lab in ("A.COUNSEL", "R.COUNSEL"):
            P[lab].add((d, int(s), int(e)))

for lab in ("A.COUNSEL", "R.COUNSEL"):
    precision, recall, f1, n_pred = prf_sets(G[lab], P[lab])
    print(
        f"{lab:10s} champion P={precision:.3f} "
        f"R={recall:.3f} F1={f1:.3f} pred={n_pred}"
    )

champion_counsel_f1 = {}
for lab in ("A.COUNSEL", "R.COUNSEL"):
    champion_counsel_f1[lab] = prf_sets(G[lab], P[lab])[2]

print("\nRule F1:", r_va)
print("Champion F1:", champion_counsel_f1)

for lab in ("A.COUNSEL", "R.COUNSEL"):
    print(
        f"{lab}: rule-minus-champion F1 = "
        f"{r_va[lab] - champion_counsel_f1[lab]:+.4f}"
    )


In [ ]:
from collections import Counter

print("=== DATA ALIGNMENT ===")
print("val_texts:", len(val_texts), "documents")
print("val_gold:", len(val_gold), "documents")
print("text ID examples:", [(repr(k), type(k).__name__) for k in list(val_texts)[:5]])
print("gold ID examples:", [(repr(k), type(k).__name__) for k in list(val_gold)[:5]])

text_ids = set(val_texts)
gold_ids = set(val_gold)

print("IDs shared directly:", len(text_ids & gold_ids))
print("Text IDs missing gold:", len(text_ids - gold_ids))
print("Gold IDs missing text:", len(gold_ids - text_ids))

print("\n=== GOLD COUNTS ===")
for lab in ("A.COUNSEL", "R.COUNSEL"):
    spans = [
        (d, int(e[0]), int(e[1]))
        for d, ents in val_gold.items()
        for e in ents
        if len(e) >= 3 and e[2] == lab
    ]
    print(lab, "gold spans:", len(spans), "examples:", spans[:5])

print("\n=== OFFSET SANITY CHECK ===")
checked = 0
for d, text in list(val_texts.items())[:3]:
    enc = tokenizer.encode(text, add_special_tokens=False)
    print(
        "doc:", repr(d),
        "text length:", len(text),
        "token count:", len(enc.ids),
        "first offsets:", enc.offsets[:8],
        "text at first offsets:",
        [text[s:e] for s, e in enc.offsets[:8] if e > s]
    )
    checked += 1

print("\n=== GOLD SPAN BOUNDS ===")
for lab in ("A.COUNSEL", "R.COUNSEL"):
    valid = invalid = 0
    for d, ents in val_gold.items():
        if d not in val_texts:
            continue
        for e in ents:
            if len(e) >= 3 and e[2] == lab:
                s, end = int(e[0]), int(e[1])
                if 0 <= s < end <= len(val_texts[d]):
                    valid += 1
                else:
                    invalid += 1
    print(lab, "valid:", valid, "invalid:", invalid)

In [ ]:
from collections import Counter, defaultdict
import torch

# Check that the expected objects exist
assert "champion_model" in globals()
assert "diag_doc_probs" in globals()
assert "BIO_LABELS" in globals()

id_to_bio = {i: label for i, label in enumerate(BIO_LABELS)}

def decode_bio_debug(offsets, pred_ids):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    for (start, end), idx in zip(offsets, pred_ids):
        if start == end:
            continue

        bio = id_to_bio[int(idx)]

        if bio == "O" or "-" not in bio:
            if current_label is not None:
                spans.append((current_start, current_end, current_label))
                current_label = None
            continue

        prefix, label = bio.split("-", 1)

        if prefix == "B" or label != current_label:
            if current_label is not None:
                spans.append((current_start, current_end, current_label))
            current_label = label
            current_start, current_end = start, end
        elif prefix == "I" and label == current_label:
            current_end = end

    if current_label is not None:
        spans.append((current_start, current_end, current_label))

    return spans

pred_counts = Counter()
pred_examples = defaultdict(list)

for doc_id, text in val_texts.items():
    offsets, probs = diag_doc_probs(champion_model, text)
    pred_ids = probs.argmax(dim=-1).tolist()

    for start, end, label in decode_bio_debug(offsets, pred_ids):
        pred_counts[label] += 1

        if label in ("A.COUNSEL", "R.COUNSEL") and len(pred_examples[label]) < 10:
            pred_examples[label].append({
                "doc_id": doc_id,
                "start": start,
                "end": end,
                "text": text[start:end]
            })

print("=== Predicted BIO label counts ===")
print(pred_counts.most_common())

print("\n=== Counsel prediction examples ===")
for label in ("A.COUNSEL", "R.COUNSEL"):
    print(f"\n{label}: {pred_counts[label]} predicted spans")
    for example in pred_examples[label]:
        print(example)

print("\n=== Gold examples ===")
for label in ("A.COUNSEL", "R.COUNSEL"):
    shown = 0
    for doc_id, entities in val_gold.items():
        text = val_texts[doc_id]
        for entity in entities:
            if len(entity) >= 3 and entity[2] == label:
                start, end = int(entity[0]), int(entity[1])
                print(label, repr(text[start:end]), (start, end))
                shown += 1
                if shown >= 5:
                    break
        if shown >= 5:
            break

In [ ]:
import re
import random
from collections import defaultdict

random.seed(1)

# Define the honorific pattern used by the original diagnostic.
HON_BEFORE = re.compile(
    r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$"
)

def side_state(text, s):
    ctx = text[max(0, s - 600):s]

    a = [m.start() for m in SIDE_A.finditer(ctx)]
    r = [m.start() for m in SIDE_R.finditer(ctx)]

    if not a and not r:
        return "none"
    if a and not r:
        return "A-cue only"
    if r and not a:
        return "R-cue only"

    return "both (nearer wins)"


def side_error_report(texts, gold, name, show=6):
    acc = defaultdict(lambda: [0, 0])
    wrong, missed = [], []

    for d, text in texts.items():
        gl = {
            (int(e[0]), int(e[1])): e[2]
            for e in gold[d]
            if len(e) >= 3 and e[2] in ("A.COUNSEL", "R.COUNSEL")
        }

        rule = set(rule_spans(text))

        # Side accuracy, but only on spans whose boundaries match gold.
        for s, e in rule:
            if (s, e) in gl:
                st = side_state(text, s)
                pred = side(text, s)
                ok = pred == gl[(s, e)]

                acc[st][0] += int(ok)
                acc[st][1] += 1

                if not ok:
                    wrong.append((
                        text[max(0, s - 160):s].replace("\n", " "),
                        text[s:e],
                        gl[(s, e)],
                        pred,
                        st
                    ))

        # Gold spans not proposed by the rules.
        for (s, e), lab in gl.items():
            if (s, e) not in rule:
                left = text[max(0, s - 40):s]
                missed.append((
                    left.replace("\n", " "),
                    text[s:e].replace("\n", " "),
                    lab,
                    bool(HON_BEFORE.search(left))
                ))

    print(f"\n=== {name}: side accuracy by cue state ===")
    for st, (ok, n) in sorted(acc.items()):
        print(f"  {st:20s} {ok:3d}/{n:3d} = {ok/max(n, 1):.3f}")

    print(f"\n--- Wrong-side examples (up to {show}) ---")
    for ctx, nm, gl_lab, pred, st in random.sample(
        wrong, min(show, len(wrong))
    ):
        print(
            f"[{st}] gold={gl_lab} pred={pred}\n"
            f"   context: ...{ctx!r}\n"
            f"   span: {nm!r}"
        )

    n_hon = sum(1 for item in missed if item[3])
    print(
        f"\nMissed gold spans: {len(missed)} total; "
        f"{n_hon} follow an honorific, "
        f"{len(missed) - n_hon} do not."
    )

    print(f"\n--- Missed gold examples (up to {show}) ---")
    for left, nm, lab, hon in random.sample(
        missed, min(show, len(missed))
    ):
        print(
            f"gold={lab} honorific_before={hon}\n"
            f"   left: ...{left!r}\n"
            f"   gold: {nm!r}"
        )

print("Definitions restored.")
print("SIDE_A available:", "SIDE_A" in globals())
print("SIDE_R available:", "SIDE_R" in globals())
print("rule_spans available:", "rule_spans" in globals())
print("side available:", "side" in globals())

In [ ]:
# Check that the original notebook's required objects exist.
required = [
    "side_error_report", "train_texts", "train_gold",
    "val_texts", "val_gold", "rule_spans", "side",
    "SIDE_A", "SIDE_R", "HON_BEFORE"
]
missing = [name for name in required if name not in globals()]

if missing:
    print("Missing variables/functions:", missing)
    print("Run the original notebook cells defining these objects first.")
else:
    side_error_report(
        train_texts, train_gold,
        "FOLD 1",
        show=10
    )

    side_error_report(
        val_texts, val_gold,
        "FOLD 2",
        show=10
    )

In [ ]:
from collections import defaultdict

def counsel_rule_diagnostics(texts, gold, name):
    gold_by_label = defaultdict(set)
    pred_by_label = defaultdict(set)

    for doc_id, entities in gold.items():
        for ent in entities:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_by_label[ent[2]].add(
                    (doc_id, int(ent[0]), int(ent[1]))
                )

    for doc_id, text in texts.items():
        for start, end in rule_spans(text):
            label = side(text, start)
            if label in ("A.COUNSEL", "R.COUNSEL"):
                pred_by_label[label].add(
                    (doc_id, int(start), int(end))
                )

    print(f"\n=== {name}: exact-span counsel rule evaluation ===")
    for label in ("A.COUNSEL", "R.COUNSEL"):
        g = gold_by_label[label]
        p = pred_by_label[label]
        correct = len(g & p)
        precision = correct / len(p) if p else 0.0
        recall = correct / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        print(
            f"{label:10s} gold={len(g):4d} predicted={len(p):4d} "
            f"matched={correct:4d} P={precision:.3f} "
            f"R={recall:.3f} F1={f1:.3f}"
        )

counsel_rule_diagnostics(train_texts, train_gold, "FOLD 1")
counsel_rule_diagnostics(val_texts, val_gold, "FOLD 2")

In [ ]:
import re
from collections import defaultdict

# Candidate patterns only: this does not assign labels or alter model weights.
INITIAL = r"(?:[A-Z]\.?\s*){1,4}"
NAME_WORD = r"[A-Z][A-Za-z'’-]+"
NAME = rf"(?:{INITIAL})?{NAME_WORD}(?:\s+{NAME_WORD}){{0,3}}"

HONORIFIC = (
    r"(?:Mr|Mrs|Ms|Miss|Dr|Shri|Sri|Smt)\.?"
)

# A name after an honorific, or a name followed by common legal-role cues.
HON_NAME_RE = re.compile(
    rf"(?<!\w){HONORIFIC}\s+({NAME})",
    flags=re.IGNORECASE
)

ROLE_CUE_RE = re.compile(
    r"\b(?:counsel\s+for|appearing\s+for|for\s+the\s+"
    r"(?:appellant|respondent|petitioner|applicant|state))\b",
    flags=re.IGNORECASE
)

def expanded_candidates(text):
    spans = set()

    for m in HON_NAME_RE.finditer(text):
        # Capture group contains only the candidate name, not the honorific.
        s, e = m.span(1)
        spans.add((s, e))

    return spans

def candidate_recall_report(texts, gold, name):
    gold_spans = set()
    old_spans = set()
    new_spans = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_spans.add((doc_id, int(ent[0]), int(ent[1])))

        for s, e in rule_spans(text):
            old_spans.add((doc_id, int(s), int(e)))

        for s, e in expanded_candidates(text):
            new_spans.add((doc_id, int(s), int(e)))

    old_hits = len(gold_spans & old_spans)
    new_hits = len(gold_spans & new_spans)
    union_hits = len(gold_spans & (old_spans | new_spans))

    print(f"\n{name}")
    print("Gold counsel spans:", len(gold_spans))
    print("Original-rule matches:", old_hits)
    print("Honorific-candidate matches:", new_hits)
    print("Union matches:", union_hits)
    print("Additional gold spans recovered:", union_hits - old_hits)

candidate_recall_report(train_texts, train_gold, "FOLD 1")

In [ ]:
def inspect_new_candidates(texts, gold, max_examples=30):
    shown = 0

    for doc_id, text in texts.items():
        gold_map = {
            (int(e[0]), int(e[1])): e[2]
            for e in gold[doc_id]
            if len(e) >= 3 and e[2] in ("A.COUNSEL", "R.COUNSEL")
        }
        old = set(rule_spans(text))

        for s, e in sorted(expanded_candidates(text)):
            if (s, e) in old:
                continue

            snippet = text[max(0, s-45):min(len(text), e+45)]
            label = gold_map.get((s, e), "NOT EXACT GOLD")

            print(
                f"doc={doc_id} gold={label}\n"
                f"  candidate={text[s:e]!r}\n"
                f"  context={snippet!r}"
            )
            shown += 1
            if shown >= max_examples:
                return

inspect_new_candidates(train_texts, train_gold)

In [ ]:
import re
from collections import Counter, defaultdict

def inspect_missed_contexts(texts, gold, show=50):
    missed = []

    for doc_id, text in texts.items():
        existing = set(rule_spans(text))

        for ent in gold[doc_id]:
            if len(ent) < 3 or ent[2] not in ("A.COUNSEL", "R.COUNSEL"):
                continue

            s, e, label = int(ent[0]), int(ent[1]), ent[2]

            if (s, e) in existing:
                continue

            left = text[max(0, s - 140):s].replace("\n", " ")
            name = text[s:e].replace("\n", " ")
            right = text[e:min(len(text), e + 70)].replace("\n", " ")

            # Broad contextual indicators, not predictions.
            cue = bool(re.search(
                r"\b(counsel|advocate|appellant|respondent|petitioner|"
                r"appearing|for the state|for the applicant|"
                r"senior counsel|learned counsel)\b",
                left,
                re.IGNORECASE
            ))

            missed.append({
                "doc": doc_id,
                "label": label,
                "name": name,
                "left": left,
                "right": right,
                "cue": cue
            })

    print("Missed gold spans:", len(missed))
    print("With counsel/party context:", sum(x["cue"] for x in missed))
    print("Without those context cues:", sum(not x["cue"] for x in missed))

    print("\n--- Missed spans with relevant context ---")
    selected = [x for x in missed if x["cue"]]
    for x in selected[:show]:
        print(
            f"\n[{x['label']}] {x['name']!r} | doc={x['doc']}\n"
            f"LEFT:  ...{x['left']!r}\n"
            f"RIGHT: {x['right']!r}"
        )

inspect_missed_contexts(train_texts, train_gold, show=30)

In [ ]:
def counsel_context_counts(texts, gold):
    patterns = {
        "for the appellant/respondent": (
            r"\bfor the (?:learned )?(?:appellant|respondent|"
            r"petitioner|applicant|complainant|accused|state)\b"
        ),
        "counsel/advocate": r"\b(?:counsel|advocate)s?\b",
        "appearing for": r"\bappearing for\b",
        "senior counsel": r"\bsenior counsel\b",
        "party heading": (
            r"\b(?:for the appellant|for the respondent|"
            r"for the petitioners?|for the state)\s*:"
        ),
        "learned counsel": r"\blearned counsel\b",
    }

    counts = Counter()
    total = Counter()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) < 3 or ent[2] not in ("A.COUNSEL", "R.COUNSEL"):
                continue

            s = int(ent[0])
            ctx = text[max(0, s - 220):s]

            for name, pattern in patterns.items():
                if re.search(pattern, ctx, re.IGNORECASE):
                    counts[name] += 1
                total[name] += 1

    print("Gold counsel spans preceded by each cue:")
    for name, count in counts.most_common():
        print(f"{name:32s} {count}")

counsel_context_counts(train_texts, train_gold)

In [ ]:
import re
from collections import defaultdict

# Candidate name: supports initials, multi-word names, and common punctuation.
INITIAL = r"(?:[A-Z]\.?\s*){1,4}"
WORD = r"[A-Z][A-Za-z'’\-]+"
NAME_PATTERN = rf"(?:{INITIAL})?{WORD}(?:\s+{WORD}){{0,3}}"

# Explicit legal representation contexts.
SECTION_RE = re.compile(
    r"\b(?:"
    r"counsel\s+for\s+(?:the\s+)?(?:appellant|respondent|petitioner|"
    r"applicant|complainant|accused|state|opposite\s+party)s?"
    r"|for\s+(?:the\s+)?(?:appellant|respondent|petitioner|applicant|"
    r"complainant|accused|state|opposite\s+party)s?"
    r"|appearing\s+for"
    r"|agent\s+for\s+(?:the\s+)?(?:appellant|respondent|petitioner|"
    r"applicant|respondent)s?"
    r")\s*[:\-]?\s*",
    re.IGNORECASE
)

# Stop at a clear section boundary to avoid collecting unrelated prose.
SECTION_STOP_RE = re.compile(
    r"\b(?:Hon['’]?ble|JUDGMENT|ORDER|CORAM|"
    r"the\s+following\s+order|judgment\s+was\s+delivered)\b",
    re.IGNORECASE
)

def counsel_list_candidates(text):
    spans = set()

    for section_match in SECTION_RE.finditer(text):
        start = section_match.end()
        limit = min(len(text), start + 500)
        region = text[start:limit]

        stop = SECTION_STOP_RE.search(region)
        if stop:
            region = region[:stop.start()]

        # Lists commonly end at a line break, a long gap, or a new heading.
        # Keep this deliberately conservative.
        region = re.split(
            r"\n\s*\n|(?:\s{3,})|"
            r"\b(?:versus|JUDGMENT|ORDER)\b",
            region,
            maxsplit=1,
            flags=re.IGNORECASE
        )[0]

        for match in re.finditer(NAME_PATTERN, region):
            s = start + match.start()
            e = start + match.end()
            candidate = text[s:e].strip()

            if not candidate:
                continue

            # Reject obvious generic/legal role words.
            if candidate.lower() in {
                "appellant", "respondent", "petitioner", "applicant",
                "state", "court", "justice", "judge", "counsel",
                "advocate", "senior counsel", "learned counsel"
            }:
                continue

            spans.add((s, e))

    return spans

In [ ]:
def evaluate_candidate_expansion(texts, gold, name):
    gold_spans = set()
    old_spans = set()
    extra_spans = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_spans.add((doc_id, int(ent[0]), int(ent[1])))

        for s, e in rule_spans(text):
            old_spans.add((doc_id, int(s), int(e)))

        for s, e in counsel_list_candidates(text):
            extra_spans.add((doc_id, int(s), int(e)))

    combined = old_spans | extra_spans

    def report(label, pred):
        tp = len(gold_spans & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_spans) if gold_spans else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        print(
            f"{label:22s} TP={tp:3d} pred={len(pred):4d} "
            f"P={p:.3f} R={r:.3f} F1={f:.3f}"
        )

    print(f"\n=== {name} ===")
    report("Original rule", old_spans)
    report("List candidates", extra_spans)
    report("Union candidates", combined)

    print("\nExamples of newly recovered gold spans:")
    recovered = sorted((gold_spans & combined) - old_spans)
    for doc_id, s, e in recovered[:20]:
        print(doc_id, repr(texts[doc_id][s:e]))

    return {
        "gold": gold_spans,
        "original": old_spans,
        "expanded": extra_spans,
        "combined": combined,
    }

fold1_candidate_eval = evaluate_candidate_expansion(
    train_texts, train_gold, "FOLD 1"
)

In [ ]:
import re

# Cues that strongly associate a nearby name with legal representation.
AFTER_NAME_CUE = re.compile(
    r"^\s*(?:,?\s*\([^)]{0,50}\s+with\s+him\)|"
    r"\s*,?\s*(?:Sr\.?\s+Adv\.?|Adv\.?|Advocate|"
    r"Solicitor[- ]General|Attorney[- ]General))?"
    r"\s*,?\s*(?:for\s+(?:the\s+)?"
    r"(?:appellant|respondent|petitioner|applicant|"
    r"complainant|accused|state|intervener)s?\b)",
    re.IGNORECASE
)

def filtered_counsel_candidates(text):
    accepted = set()

    for s, e in counsel_list_candidates(text):
        # Inspect a short window after the name.
        after = text[e:min(len(text), e + 100)]

        # The original list-candidate function is deliberately broad;
        # accept only candidates followed by a representation cue.
        if AFTER_NAME_CUE.search(after):
            accepted.add((s, e))

    return accepted


def evaluate_filtered_expansion(texts, gold, name):
    gold_spans = set()
    original = set()
    filtered = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_spans.add((doc_id, int(ent[0]), int(ent[1])))

        original.update(
            (doc_id, int(s), int(e)) for s, e in rule_spans(text)
        )
        filtered.update(
            (doc_id, int(s), int(e))
            for s, e in filtered_counsel_candidates(text)
        )

    combined = original | filtered

    def score(pred):
        tp = len(gold_spans & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_spans) if gold_spans else 0.0
        f = 2*p*r/(p+r) if p+r else 0.0
        return tp, len(pred), p, r, f

    for label, pred in [
        ("Original", original),
        ("Filtered only", filtered),
        ("Combined", combined)
    ]:
        tp, n, p, r, f = score(pred)
        print(
            f"{label:15s} TP={tp:3d} pred={n:4d} "
            f"P={p:.3f} R={r:.3f} F1={f:.3f}"
        )

evaluate_filtered_expansion(train_texts, train_gold, "FOLD 1")

In [ ]:
# Inspect new candidates introduced by the filtered expansion.
shown = 0

for doc_id, text in train_texts.items():
    old = {(int(s), int(e)) for s, e in rule_spans(text)}
    new = filtered_counsel_candidates(text)

    gold_map = {
        (int(ent[0]), int(ent[1])): ent[2]
        for ent in train_gold[doc_id]
        if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL")
    }

    for s, e in sorted(new - old):
        print(
            f"doc={doc_id} "
            f"gold={gold_map.get((s, e), 'NOT EXACT GOLD')}\n"
            f"  candidate={text[s:e]!r}\n"
            f"  context={text[max(0, s-90):min(len(text), e+100)]!r}"
        )
        shown += 1
        if shown >= 30:
            break

    if shown >= 30:
        break

if shown == 0:
    print("No additional candidates were introduced.")

In [ ]:
import re

BAD_EXACT = {
    "india", "agent", "panel lawyer", "advocate", "counsel",
    "senior counsel", "learned counsel", "solicitor general",
    "attorney general", "justice", "judge", "court", "state",
    "appellant", "respondent", "petitioner", "applicant",
    "intervener", "for the appellant", "for the respondent",
}

BAD_TRAILING = re.compile(
    r"\s+(?:counsel|advocate|lawyer|justice|judge|court|"
    r"submitted|contended|argued|appearing|for|in|on|also|"
    r"were|was|has|have|and|or|the|his|her|their)\s*$",
    re.IGNORECASE
)

def clean_candidate_span(text, s, e):
    candidate = text[s:e].strip()

    if candidate.lower() in BAD_EXACT:
        return None

    # Remove a trailing generic/legal word only if it is at the end.
    while candidate and BAD_TRAILING.search(candidate):
        candidate = BAD_TRAILING.sub("", candidate).rstrip()

    # Avoid candidates that are clearly not personal names.
    if not candidate or candidate.lower() in BAD_EXACT:
        return None

    # Require at least one alphabetic character.
    if not re.search(r"[A-Za-z]", candidate):
        return None

    # Recover the trimmed character offsets.
    left_trim = len(text[s:e]) - len(text[s:e].lstrip())
    new_s = s + left_trim
    new_e = new_s + len(candidate)

    return new_s, new_e


def cleaned_counsel_candidates(text):
    cleaned = set()

    for s, e in filtered_counsel_candidates(text):
        span = clean_candidate_span(text, s, e)
        if span is not None:
            cleaned.add(span)

    return cleaned


def evaluate_cleaned_expansion(texts, gold, name):
    gold_spans = set()
    original = set()
    cleaned = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_spans.add((doc_id, int(ent[0]), int(ent[1])))

        original.update(
            (doc_id, int(s), int(e)) for s, e in rule_spans(text)
        )
        cleaned.update(
            (doc_id, int(s), int(e))
            for s, e in cleaned_counsel_candidates(text)
        )

    def report(label, pred):
        tp = len(gold_spans & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_spans) if gold_spans else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        print(
            f"{label:18s} TP={tp:3d} pred={len(pred):4d} "
            f"P={p:.3f} R={r:.3f} F1={f1:.3f}"
        )

    combined = original | cleaned

    print(f"\n=== {name} ===")
    report("Original", original)
    report("Cleaned candidates", cleaned)
    report("Combined", combined)

evaluate_cleaned_expansion(
    train_texts, train_gold, "FOLD 1"
)

In [ ]:
def evaluate_rule_union(texts, gold, name):
    gold_by_label = defaultdict(set)
    original_by_label = defaultdict(set)
    combined_by_label = defaultdict(set)

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_by_label[ent[2]].add(
                    (doc_id, int(ent[0]), int(ent[1]))
                )

        # Existing rules, with the original side assignment.
        for s, e in rule_spans(text):
            lab = side(text, s)
            if lab in ("A.COUNSEL", "R.COUNSEL"):
                original_by_label[lab].add((doc_id, int(s), int(e)))
                combined_by_label[lab].add((doc_id, int(s), int(e)))

        # Add only cleaned candidates not already captured by the original rules.
        for s, e in cleaned_counsel_candidates(text):
            lab = side(text, s)
            if lab in ("A.COUNSEL", "R.COUNSEL"):
                combined_by_label[lab].add((doc_id, int(s), int(e)))

    def report(title, pred_by_label):
        # Micro-aggregate exact spans across both counsel labels.
        g = gold_by_label["A.COUNSEL"] | gold_by_label["R.COUNSEL"]
        p = pred_by_label["A.COUNSEL"] | pred_by_label["R.COUNSEL"]
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        print(
            f"{title:18s} TP={tp:3d} pred={len(p):4d} "
            f"P={precision:.3f} R={recall:.3f} F1={f1:.3f}"
        )

        for lab in ("A.COUNSEL", "R.COUNSEL"):
            gl = gold_by_label[lab]
            pl = pred_by_label[lab]
            hit = len(gl & pl)
            pp = hit / len(pl) if pl else 0.0
            rr = hit / len(gl) if gl else 0.0
            ff = 2 * pp * rr / (pp + rr) if pp + rr else 0.0
            print(
                f"  {lab:10s} TP={hit:3d} pred={len(pl):4d} "
                f"P={pp:.3f} R={rr:.3f} F1={ff:.3f}"
            )

    print(f"\n=== {name} ===")
    report("Original", original_by_label)
    report("Cleaned + original", combined_by_label)

evaluate_rule_union(val_texts, val_gold, "FOLD 2 HELD-OUT")

In [ ]:
from collections import defaultdict

def compare_labeled_counsel(texts, gold, name):
    gold_labeled = set()
    original_labeled = set()
    combined_labeled = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_labeled.add(
                    (doc_id, int(ent[0]), int(ent[1]), ent[2])
                )

        for s, e in rule_spans(text):
            lab = side(text, s)
            if lab in ("A.COUNSEL", "R.COUNSEL"):
                item = (doc_id, int(s), int(e), lab)
                original_labeled.add(item)
                combined_labeled.add(item)

        for s, e in cleaned_counsel_candidates(text):
            lab = side(text, s)
            if lab in ("A.COUNSEL", "R.COUNSEL"):
                combined_labeled.add((doc_id, int(s), int(e), lab))

    def show(label, pred):
        tp = len(gold_labeled & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_labeled) if gold_labeled else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        print(
            f"{label:18s} exact labeled TP={tp:3d} "
            f"pred={len(pred):4d} P={p:.3f} R={r:.3f} F1={f1:.3f}"
        )

    print(f"\n=== {name}: exact labeled spans ===")
    show("Original", original_labeled)
    show("Combined", combined_labeled)
    print("New labeled predictions:", len(combined_labeled - original_labeled))
    print("New exact labeled matches:", len(
        (gold_labeled & combined_labeled) -
        (gold_labeled & original_labeled)
    ))

compare_labeled_counsel(val_texts, val_gold, "FOLD 2")

In [ ]:
def inspect_new_labeled_candidates(texts, gold, limit=40):
    shown = 0

    for doc_id, text in texts.items():
        old = {(int(s), int(e)) for s, e in rule_spans(text)}

        gold_map = {
            (int(ent[0]), int(ent[1])): ent[2]
            for ent in gold[doc_id]
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL")
        }

        for s, e in sorted(cleaned_counsel_candidates(text)):
            if (s, e) in old:
                continue

            pred = side(text, s)
            gold_label = gold_map.get((s, e), "NO EXACT MATCH")

            print(
                f"doc={doc_id} gold={gold_label} pred={pred}\n"
                f"  span={text[s:e]!r}\n"
                f"  context={text[max(0, s-180):min(len(text), e+120)]!r}\n"
            )

            shown += 1
            if shown >= limit:
                return

inspect_new_labeled_candidates(val_texts, val_gold, limit=40)

In [ ]:
import re
from collections import defaultdict

# Explicit representation phrases.
EXPLICIT_A = re.compile(
    r"\bfor\s+(?:the\s+)?(?:appellant|petitioner|applicant|"
    r"complainant|plaintiff)s?\b",
    re.IGNORECASE
)

EXPLICIT_R = re.compile(
    r"\bfor\s+(?:the\s+)?(?:respondent|accused|defendant|"
    r"opposite\s+party|state)s?\b",
    re.IGNORECASE
)

def explicit_side(text, start, end):
    """
    Look for a side cue immediately following the candidate, allowing
    other names in the same list to precede the cue.
    Returns A.COUNSEL, R.COUNSEL, or None.
    """
    # First inspect the next 100 chars, where list-ending cues commonly occur.
    right = text[end:min(len(text), end + 100)]

    # If a side cue occurs before a strong sentence/section boundary, use it.
    boundary = re.search(r"[.;\n]| {3,}", right)
    if boundary:
        right = right[:boundary.start()]

    a = EXPLICIT_A.search(right)
    r = EXPLICIT_R.search(right)

    if a and not r:
        return "A.COUNSEL"
    if r and not a:
        return "R.COUNSEL"

    # No unambiguous local explicit cue: preserve the original heuristic.
    return side(text, start)


def evaluate_side_variant(texts, gold, name):
    gold_labeled = set()
    original_labeled = set()
    explicit_labeled = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_labeled.add(
                    (doc_id, int(ent[0]), int(ent[1]), ent[2])
                )

        candidates = set(rule_spans(text)) | set(cleaned_counsel_candidates(text))

        for s, e in candidates:
            original_label = side(text, s)
            if original_label in ("A.COUNSEL", "R.COUNSEL"):
                original_labeled.add((doc_id, int(s), int(e), original_label))

            new_label = explicit_side(text, s, e)
            if new_label in ("A.COUNSEL", "R.COUNSEL"):
                explicit_labeled.add((doc_id, int(s), int(e), new_label))

    def report(label, pred):
        tp = len(gold_labeled & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_labeled) if gold_labeled else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        print(
            f"{label:20s} TP={tp:3d} pred={len(pred):4d} "
            f"P={p:.3f} R={r:.3f} F1={f1:.3f}"
        )

    print(f"\n=== {name}: labeled counsel spans ===")
    report("Original side()", original_labeled)
    report("Explicit side cue", explicit_labeled)


evaluate_side_variant(train_texts, train_gold, "FOLD 1")
evaluate_side_variant(val_texts, val_gold, "FOLD 2")

In [ ]:
def counsel_side_ablation(texts, gold, name):
    gold_labeled = set()
    original_candidates = set()
    expanded_candidates_set = set()

    for doc_id, entities in gold.items():
        for ent in entities:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_labeled.add(
                    (doc_id, int(ent[0]), int(ent[1]), ent[2])
                )

    for doc_id, text in texts.items():
        for s, e in rule_spans(text):
            original_candidates.add((doc_id, int(s), int(e)))

        for s, e in cleaned_counsel_candidates(text):
            expanded_candidates_set.add((doc_id, int(s), int(e)))

    def make_labeled(candidates, side_fn):
        pred = set()

        for doc_id, s, e in candidates:
            text = texts[doc_id]
            lab = side_fn(text, s, e)

            if lab in ("A.COUNSEL", "R.COUNSEL"):
                pred.add((doc_id, s, e, lab))

        return pred

    def score(pred):
        tp = len(gold_labeled & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_labeled) if gold_labeled else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        return tp, len(pred), p, r, f1

    combined_candidates = original_candidates | expanded_candidates_set

    variants = [
        ("Original candidates + side", original_candidates,
         lambda t, s, e: side(t, s)),
        ("Original candidates + explicit", original_candidates,
         explicit_side),
        ("Expanded candidates + side", combined_candidates,
         lambda t, s, e: side(t, s)),
        ("Expanded + explicit", combined_candidates,
         explicit_side),
    ]

    print(f"\n=== {name}: candidate/side ablation ===")
    for label, candidates, fn in variants:
        tp, n, p, r, f1 = score(make_labeled(candidates, fn))
        print(
            f"{label:32s} TP={tp:3d} pred={n:4d} "
            f"P={p:.3f} R={r:.3f} F1={f1:.3f}"
        )

counsel_side_ablation(train_texts, train_gold, "FOLD 1")
counsel_side_ablation(val_texts, val_gold, "FOLD 2")

In [ ]:
def report_explicit_side_by_label(texts, gold, name):
    gold_by_label = defaultdict(set)
    pred_by_label = defaultdict(set)

    for doc_id, entities in gold.items():
        for ent in entities:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_by_label[ent[2]].add(
                    (doc_id, int(ent[0]), int(ent[1]))
                )

    for doc_id, text in texts.items():
        # Use the original candidate generator only.
        for s, e in rule_spans(text):
            label = explicit_side(text, s, e)
            if label in ("A.COUNSEL", "R.COUNSEL"):
                pred_by_label[label].add((doc_id, int(s), int(e)))

    print(f"\n=== {name}: original candidates + explicit side ===")
    for label in ("A.COUNSEL", "R.COUNSEL"):
        g, p = gold_by_label[label], pred_by_label[label]
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        print(
            f"{label:10s} gold={len(g):3d} pred={len(p):3d} "
            f"TP={tp:3d} P={precision:.3f} R={recall:.3f} F1={f1:.3f}"
        )

report_explicit_side_by_label(train_texts, train_gold, "FOLD 1")
report_explicit_side_by_label(val_texts, val_gold, "FOLD 2")

In [ ]:
import os
import json
import glob
from collections import defaultdict
from sklearn.metrics import f1_score

# 1. Locate the saved champion threshold file
threshold_files = []
for root in ("/kaggle/working", "/kaggle/input"):
    for path in glob.glob(
        os.path.join(root, "**", "*threshold*.json"),
        recursive=True
    ):
        threshold_files.append(path)

threshold_files = sorted(set(threshold_files))
print("Threshold JSON files found:")
for path in threshold_files:
    print(" ", path)

# 2. Load the recovery thresholds if champ_thr is missing
if "champ_thr" not in globals():
    candidates = [
        p for p in threshold_files
        if "champion" in os.path.basename(p).lower()
    ]

    if not candidates:
        raise FileNotFoundError(
            "No champion threshold JSON found. "
            "Check that the recovery dataset is attached to this Kaggle notebook."
        )

    print("\nInspecting candidate threshold files:")
    for path in candidates:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        print("\nFile:", path)
        print("JSON type:", type(obj).__name__)
        if isinstance(obj, dict):
            print("Keys:", list(obj.keys())[:40])

    # Prefer a directly usable label -> threshold dictionary.
    loaded = None
    for path in candidates:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, dict):
            for key in ("champ_thr", "thresholds", "label_thresholds"):
                if key in obj and isinstance(obj[key], dict):
                    loaded = obj[key]
                    break

            if loaded is None and obj and all(
                isinstance(v, (int, float)) for v in obj.values()
            ):
                loaded = obj

        if loaded is not None:
            print("\nUsing thresholds from:", path)
            break

    if loaded is None:
        raise ValueError(
            "Found threshold JSON files, but couldn't safely identify "
            "a label-to-threshold mapping. Share the printed keys."
        )

    champ_thr = loaded

print("\nChampion thresholds loaded:")
print(champ_thr)

# 3. Define an exact-span macro-F1 evaluator.
# Gold entities are expected to be (start, end, label) or similar triples.
def score(predictions, gold=None):
    if gold is None or not gold:
        if "val_gold" not in globals():
            raise RuntimeError("val_gold is not available in this kernel.")
        gold = val_gold

    labels = list(BIO_LABELS) if "BIO_LABELS" in globals() else None
    if labels is None:
        labels = sorted({
            str(ent[2])
            for ents in gold.values()
            for ent in ents
            if len(ent) >= 3
        })

    gold_set = set()
    pred_set = set()

    for doc_id, entities in gold.items():
        for ent in entities:
            if len(ent) >= 3:
                gold_set.add((doc_id, int(ent[0]), int(ent[1]), str(ent[2])))

    for doc_id, entities in predictions.items():
        for ent in entities:
            if len(ent) >= 3:
                pred_set.add((doc_id, int(ent[0]), int(ent[1]), str(ent[2])))

    # Convert span sets to per-label exact-match counts.
    per_label = {}
    for label in labels:
        g = {x for x in gold_set if x[3] == label}
        p = {x for x in pred_set if x[3] == label}
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        per_label[label] = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

    macro = sum(per_label.values()) / len(per_label) if per_label else 0.0
    return macro, per_label

print("\nEvaluator defined. No predictions or model weights were changed.")

In [ ]:
import json
import os

threshold_path = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-persistent-ner-checkpoints/champion_thresholds.json"
)
recovery_path = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-working-backup-20261010-215243/"
    "cs781_recovery/champion_thresholds_recovery.json"
)

with open(threshold_path, "r", encoding="utf-8") as f:
    champion_config = json.load(f)

print("=== CHAMPION CONFIGURATION ===")
print("Checkpoint:", champion_config.get("checkpoint"))
print("Decoder:", champion_config.get("decoder"))
print("Saved validation macro-F1:", champion_config.get("validation_macro_f1"))
print("Thresholds match loaded champ_thr:",
      champion_config.get("thresholds") == champ_thr)

if os.path.exists(recovery_path):
    with open(recovery_path, "r", encoding="utf-8") as f:
        recovery_thresholds = json.load(f)
    print("\n=== RECOVERY THRESHOLDS ===")
    print(recovery_thresholds)
    print("Recovery thresholds match champion thresholds:",
          recovery_thresholds == champ_thr)

print("\n=== REQUIRED INFERENCE FUNCTIONS ===")
for name in (
    "diag_doc_probs", "bio_viterbi_decode", "decode_spans",
    "predict_texts", "apply_thr", "as_scored",
    "score", "val_texts", "val_gold",
    "champion_model", "champ_thr"
):
    print(f"{name:24s}", "AVAILABLE" if name in globals() else "MISSING")

print("\nNo model weights or prediction files were modified.")

In [ ]:
import json
import pandas as pd

with open(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-persistent-ner-checkpoints/champion_thresholds.json",
    "r",
    encoding="utf-8"
) as f:
    champion_config = json.load(f)

with open(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-working-backup-20261010-215243/"
    "cs781_recovery/champion_thresholds_recovery.json",
    "r",
    encoding="utf-8"
) as f:
    recovery_thr = json.load(f)

champion_thr = champion_config["thresholds"]

comparison = pd.DataFrame({
    "champion_threshold": champion_thr,
    "recovery_threshold": recovery_thr
})
comparison["difference"] = (
    comparison["recovery_threshold"] - comparison["champion_threshold"]
)
comparison["changed"] = (
    comparison["champion_threshold"] != comparison["recovery_threshold"]
)

print("Saved champion validation macro-F1:",
      champion_config.get("validation_macro_f1"))
print("\nThreshold comparison:")
print(comparison.round(3).to_string())
print("\nLabels with different thresholds:",
      comparison.index[comparison["changed"]].tolist())
print("\nNo thresholds, model weights, or prediction files were changed.")

In [ ]:
import torch
import pandas as pd
from bisect import bisect_left, bisect_right

required = [
    "champion_model", "diag_doc_probs", "bio_viterbi_decode",
    "decode_spans", "val_texts", "val_gold", "champ_thr"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError("Missing definitions: " + ", ".join(missing))

# Recovery thresholds were loaded in the previous cell.
if "recovery_thr" not in globals():
    raise RuntimeError(
        "recovery_thr is missing. Rerun the threshold-comparison cell first."
    )

@torch.inference_mode()
def infer_spans_for_eval(model, texts):
    result = {}
    model.eval()

    for doc_id, text in texts.items():
        offsets, probs = diag_doc_probs(model, text)

        if len(offsets) == 0:
            result[doc_id] = []
            continue

        pred_ids = torch.as_tensor(
            bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))),
            dtype=torch.long
        ).tolist()

        starts = [int(a) for a, b in offsets]
        ends = [int(b) for a, b in offsets]
        spans = []

        for s, e, label in decode_spans(text, offsets, pred_ids):
            token_ids = [
                i for i in range(
                    bisect_right(ends, int(s)),
                    bisect_left(starts, int(e))
                )
                if ends[i] > starts[i]
            ]

            confidence = (
                float(probs[token_ids, torch.as_tensor(pred_ids)[token_ids]].mean())
                if token_ids else 0.0
            )
            spans.append((int(s), int(e), str(label), confidence))

        result[doc_id] = spans

    return result


def threshold_spans(raw, thresholds):
    return {
        doc_id: [
            (s, e, label)
            for s, e, label, confidence in entities
            if confidence >= float(thresholds.get(label, 0.0))
        ]
        for doc_id, entities in raw.items()
    }


def exact_span_report(predictions, gold):
    labels = sorted({
        str(ent[2])
        for entities in gold.values()
        for ent in entities
        if len(ent) >= 3
    } | {
        str(ent[2])
        for entities in predictions.values()
        for ent in entities
        if len(ent) >= 3
    })

    gold_set = {
        (doc_id, int(ent[0]), int(ent[1]), str(ent[2]))
        for doc_id, entities in gold.items()
        for ent in entities if len(ent) >= 3
    }
    pred_set = {
        (doc_id, int(ent[0]), int(ent[1]), str(ent[2]))
        for doc_id, entities in predictions.items()
        for ent in entities if len(ent) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gold_set if x[3] == label}
        p = {x for x in pred_set if x[3] == label}
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows.append({
            "label": label,
            "gold": len(g),
            "predicted": len(p),
            "TP": tp,
            "precision": precision,
            "recall": recall,
            "F1": f1
        })

    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report


# Run model inference once; threshold comparisons reuse these outputs.
raw_val_predictions = infer_spans_for_eval(champion_model, val_texts)

champion_predictions = threshold_spans(raw_val_predictions, champ_thr)
recovery_predictions = threshold_spans(raw_val_predictions, recovery_thr)

champion_macro_eval, champion_report_eval = exact_span_report(
    champion_predictions, val_gold
)
recovery_macro_eval, recovery_report_eval = exact_span_report(
    recovery_predictions, val_gold
)

print("=== EXACT-SPAN EVALUATION ON val_gold ===")
print(f"Champion thresholds macro-F1: {champion_macro_eval:.4f}")
print(f"Recovery thresholds macro-F1: {recovery_macro_eval:.4f}")
print(f"Difference (recovery - champion): "
      f"{recovery_macro_eval - champion_macro_eval:+.4f}")

print("\n=== PER-LABEL F1 ===")
comparison = pd.DataFrame({
    "champion_F1": champion_report_eval["F1"],
    "recovery_F1": recovery_report_eval["F1"],
    "champion_predicted": champion_report_eval["predicted"],
    "recovery_predicted": recovery_report_eval["predicted"],
}).round(3)
print(comparison.to_string())

print("\nNo training, threshold-file edits, or prediction-file writes occurred.")

In [ ]:
# CS781 — Restore V5 counsel helpers from the latest notebook
# No training, checkpoint edits, or file writes.

import re
import pandas as pd
import torch
from bisect import bisect_left, bisect_right

required = [
    "champion_model", "champ_thr", "diag_doc_probs",
    "bio_viterbi_decode", "decode_spans",
    "val_texts", "val_gold", "train_texts", "train_gold",
    "rule_spans", "score"
]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(
        "Still missing foundational definitions: " + ", ".join(missing)
    )

# ---- Exact V5 side-assignment setup from the latest notebook ----
PARTY_A = r"(?:petitioners?|appellants?|applicants?|plaintiffs?|complainants?|prosecution|revisionists?)"
PARTY_R = r"(?:respondents?|defendants?|accused|non-\s*applicants?|opposite\s+part(?:y|ies))"
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"

CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s\.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s\.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")

A_, R_ = "A.COUNSEL", "R.COUNSEL"
ORDER = ["hdr", "right", "left", "inherit"]

def _cues(text, lo, hi):
    out = [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]
    out += [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]
    return out

def assign_side(text, s, e, order, prev, W_L=300, W_R=200):
    for step in order:
        if step == "right":
            cues = _cues(text, e, min(len(text), e + W_R))
            if cues:
                return min(cues)[2], "right"
        elif step == "left":
            cues = _cues(text, max(0, s - W_L), s)
            cues = [x for x in cues if x[1] <= s]
            if cues:
                return max(cues, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail):
                return A_, "hdr"
            if HDR_R.search(tail):
                return R_, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_, "default"

def counsel_spans(text, policy):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        lab, src = assign_side(text, s, e, ORDER, prev)
        prev = (e, lab, src)
        if src == "default":
            if policy == "drop":
                continue
            if policy == "A":
                lab = A_
        out.append((int(s), int(e), lab))
    return out

def counsel_f1(texts, gold, policy):
    G = {A_: set(), R_: set()}
    P = {A_: set(), R_: set()}

    for d, ents in gold.items():
        for ent in ents:
            if len(ent) >= 3 and ent[2] in G:
                G[ent[2]].add((d, int(ent[0]), int(ent[1])))

    for d, text in texts.items():
        for s, e, lab in counsel_spans(text, policy):
            P[lab].add((d, s, e))

    result = {}
    for lab in (A_, R_):
        tp = len(G[lab] & P[lab])
        p = tp / max(len(P[lab]), 1)
        r = tp / max(len(G[lab]), 1)
        result[lab] = 2*p*r/(p+r) if p+r else 0.0
    return result

# Choose fallback policy using fold 1 only.
fold1_policy_scores = {
    p: counsel_f1(train_texts, train_gold, p)
    for p in ("R", "A", "drop")
}
chosen = max(
    fold1_policy_scores,
    key=lambda p: (
        fold1_policy_scores[p][A_] + fold1_policy_scores[p][R_],
        p == "R"
    )
)

print("Fold-1 counsel policy selection:")
for p, f in fold1_policy_scores.items():
    print(f"{p:5s} A={f[A_]:.3f} R={f[R_]:.3f} "
          f"mean={(f[A_]+f[R_])/2:.3f}")
print("Selected policy:", chosen)

# Reconstruct champion inference helpers.
@torch.inference_mode()
def predict_texts(model, texts):
    result = {}
    model.eval()

    for doc_id, text in texts.items():
        offsets, probs = diag_doc_probs(model, text)
        if len(offsets) == 0:
            result[doc_id] = []
            continue

        pred_ids = torch.as_tensor(
            bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))),
            dtype=torch.long
        )
        starts = [a for a, b in offsets]
        ends = [b for a, b in offsets]
        spans = []

        for s, e, lab in decode_spans(text, offsets, pred_ids.tolist()):
            ids = [
                i for i in range(bisect_right(ends, s), bisect_left(starts, e))
                if ends[i] > starts[i]
            ]
            conf = float(probs[ids, pred_ids[ids]].mean()) if ids else 0.0
            spans.append((int(s), int(e), lab, conf))

        result[doc_id] = spans
    return result

def apply_thr(raw, thresholds):
    return {
        d: [(s, e, lab) for s, e, lab, conf in spans
            if conf >= thresholds.get(lab, 0.0)]
        for d, spans in raw.items()
    }

def as_scored(predictions):
    return {
        d: [(s, e, lab, 1.0) for s, e, lab in spans]
        for d, spans in predictions.items()
    }

def merge_policy(texts, champion_filtered, policy):
    result = {}
    for d, text in texts.items():
        rules = counsel_spans(text, policy)
        kept = [
            ent for ent in champion_filtered[d]
            if ent[2] not in (A_, R_)
        ]
        kept = [
            ent for ent in kept
            if not any(
                ent[0] < re_ and ent[1] > rs
                for rs, re_, _ in rules
            )
        ]
        result[d] = sorted(set(kept + rules))
    return result

# Evaluate once on fold 2.
raw_fold2 = predict_texts(champion_model, val_texts)
c_filt = apply_thr(raw_fold2, champ_thr)

base_macro, base_f1 = score(as_scored(c_filt), {})
counsel_predictions = merge_policy(val_texts, c_filt, chosen)
counsel_macro, counsel_f1 = score(as_scored(counsel_predictions), {})

print("\nFOLD-2 FULL-NER RESULT")
print(f"Champion:         {base_macro:.4f}")
print(f"Champion + V5:    {counsel_macro:.4f}")
print(f"Delta:            {counsel_macro - base_macro:+.4f}")
print("\nPer-label F1:")
print(pd.DataFrame({
    "champion": base_f1,
    "champion_plus_V5": counsel_f1
}).round(3).to_string())

print("\nNo model weights or prediction files were modified.")

In [ ]:
# CS781 — Diagnose why every BIO-label F1 is zero

def inspect_sample(name, obj):
    print(f"\n=== {name} ===")
    print("Container type:", type(obj).__name__)

    if not isinstance(obj, dict):
        print("Sample:", repr(obj)[:800])
        return

    print("Number of documents:", len(obj))
    for doc_id, entities in list(obj.items())[:2]:
        print("Document ID:", repr(doc_id))
        print("Value type:", type(entities).__name__)
        print("Value sample:", repr(entities)[:1000])

inspect_sample("val_gold", val_gold)
inspect_sample("champion filtered predictions", c_filt)
inspect_sample("counsel predictions", counsel_predictions)

print("\n=== SCORE FUNCTION SOURCE ===")
import inspect
try:
    print(inspect.getsource(score))
except (OSError, TypeError):
    print("Python source unavailable for score().")

print("\n=== SCORE FUNCTION MODULE ===")
print(getattr(score, "__module__", None))
print("\nNo files or model weights were modified.")

In [ ]:
# CS781 — Correct exact-span evaluator for entity-level labels
# Uses val_gold's entity labels, not BIO token tags.

import pandas as pd

def score_entity_spans(predictions, gold):
    labels = sorted({
        str(ent[2])
        for entities in gold.values()
        for ent in entities
        if len(ent) >= 3
    })

    gold_set = {
        (str(doc_id), int(ent[0]), int(ent[1]), str(ent[2]))
        for doc_id, entities in gold.items()
        for ent in entities
        if len(ent) >= 3
    }

    pred_set = {
        (str(doc_id), int(ent[0]), int(ent[1]), str(ent[2]))
        for doc_id, entities in predictions.items()
        for ent in entities
        if len(ent) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gold_set if x[3] == label}
        p = {x for x in pred_set if x[3] == label}
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows.append({
            "label": label,
            "gold": len(g),
            "predicted": len(p),
            "TP": tp,
            "precision": precision,
            "recall": recall,
            "F1": f1,
        })

    report = pd.DataFrame(rows).set_index("label")
    macro = float(report["F1"].mean()) if len(report) else 0.0
    return macro, report


# Re-evaluate existing predictions with the correct entity label set.
base_macro_fixed, base_report_fixed = score_entity_spans(c_filt, val_gold)
counsel_macro_fixed, counsel_report_fixed = score_entity_spans(
    counsel_predictions, val_gold
)

print("=== CORRECTED ENTITY-LEVEL FOLD-2 RESULTS ===")
print(f"Champion:         {base_macro_fixed:.4f}")
print(f"Champion + V5:    {counsel_macro_fixed:.4f}")
print(f"Delta:            {counsel_macro_fixed - base_macro_fixed:+.4f}")

print("\n=== PER-LABEL F1 ===")
print(pd.DataFrame({
    "champion_F1": base_report_fixed["F1"],
    "V5_F1": counsel_report_fixed["F1"],
    "champion_predicted": base_report_fixed["predicted"],
    "V5_predicted": counsel_report_fixed["predicted"],
}).round(3).to_string())

print("\nNo model weights or prediction files were modified.")

In [ ]:

# CS781 — Fold-1 cross-validation for V5
# Exact entity-span scoring; no training or checkpoint modification.

import inspect
import pandas as pd

# 1. Locate fold-1 texts and gold spans.
def first_existing(names):
    for name in names:
        if name in globals() and globals()[name] is not None:
            return name, globals()[name]
    return None, None

text_name, fold1_texts = first_existing([
    "fold1_texts", "fold1_val_texts", "val_texts_fold1",
    "texts_fold1", "f1_texts", "train_texts_fold1"
])

gold_name, fold1_gold = first_existing([
    "fold1_gold", "fold1_val_gold", "val_gold_fold1",
    "gold_fold1", "f1_gold", "train_gold_fold1"
])

print("Fold-1 text variable:", text_name)
print("Fold-1 gold variable:", gold_name)

if fold1_texts is None or fold1_gold is None:
    print("\nCould not safely identify both fold-1 inputs.")
    print("Available likely variables:")
    for name in sorted(globals()):
        if any(k in name.lower() for k in ("fold1", "fold_1", "gold", "text")):
            obj = globals()[name]
            if isinstance(obj, (dict, list, tuple)):
                print(f"  {name}: {type(obj).__name__}, len={len(obj)}")
    raise RuntimeError(
        "No evaluation was run. Use the printed variable names to map "
        "the actual fold-1 texts and gold spans, then rerun this cell."
    )

if not isinstance(fold1_texts, dict) or not isinstance(fold1_gold, dict):
    raise TypeError(
        "Expected fold-1 texts and gold spans as dictionaries keyed by document ID. "
        f"Got texts={type(fold1_texts).__name__}, "
        f"gold={type(fold1_gold).__name__}."
    )

if set(fold1_texts) != set(fold1_gold):
    raise ValueError(
        "Fold-1 text and gold document IDs do not match. "
        f"texts={len(fold1_texts)}, gold={len(fold1_gold)}."
    )

# 2. Exact-span scorer using entity labels (not B-/I- BIO tags).
def exact_entity_score(predictions, gold):
    labels = sorted({
        str(e[2])
        for ents in gold.values()
        for e in ents
        if len(e) >= 3
    })

    gs = {
        (str(doc), int(e[0]), int(e[1]), str(e[2]))
        for doc, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    ps = {
        (str(doc), int(e[0]), int(e[1]), str(e[2]))
        for doc, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gs if x[3] == label}
        p = {x for x in ps if x[3] == label}
        tp = len(g & p)
        prec = tp / len(p) if p else 0.0
        rec = tp / len(g) if g else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        rows.append({
            "label": label, "gold": len(g), "predicted": len(p),
            "TP": tp, "precision": prec, "recall": rec, "F1": f1
        })

    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report

# 3. Generate baseline predictions with the existing champion.
required = ["predict_texts", "apply_thr", "champion_model", "champ_thr",
            "counsel_spans", "merge_policy"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required notebook functions/variables: {missing}")

print("\nmerge_policy signature:", inspect.signature(merge_policy))
print("counsel_spans signature:", inspect.signature(counsel_spans))

fold1_base = apply_thr(
    predict_texts(champion_model, fold1_texts),
    champ_thr
)

# 4. Determine how this notebook's merge_policy expects its inputs.
# Do not guess the argument order: stop if it differs from the known call.
sig = inspect.signature(merge_policy)
params = list(sig.parameters)
print("merge_policy parameters:", params)

if len(params) != 3:
    raise RuntimeError(
        "The merge_policy signature is not the expected three-argument form. "
        "No V5 evaluation was run; share the printed signature so the call "
        "can be matched to this notebook."
    )

# The notebook's helper is expected to merge baseline spans with counsel spans.
# Try the common (base_predictions, texts, policy) signature first.
try:
    fold1_v5 = merge_policy(fold1_base, fold1_texts, "drop")
except Exception as exc:
    raise RuntimeError(
        "Baseline predictions were generated, but the V5 merge call failed. "
        "No result is reported. Share the merge_policy signature and error."
    ) from exc

# 5. Score both predictions against fold-1 gold spans.
base_f1, base_report = exact_entity_score(fold1_base, fold1_gold)
v5_f1, v5_report = exact_entity_score(fold1_v5, fold1_gold)

print("\n=== FOLD-1 FULL ENTITY-LEVEL RESULTS ===")
print(f"Champion:       {base_f1:.4f}")
print(f"Champion + V5:  {v5_f1:.4f}")
print(f"Delta:          {v5_f1 - base_f1:+.4f}")

comparison = pd.DataFrame({
    "champion_F1": base_report["F1"],
    "V5_F1": v5_report["F1"],
    "champion_predicted": base_report["predicted"],
    "V5_predicted": v5_report["predicted"],
})
print("\n=== PER-LABEL RESULTS ===")
print(comparison.round(3).to_string())

print("\nNo model weights or prediction files were modified.")


In [ ]:

import inspect

print("=== FOLD-1 EVALUATION METADATA ===")
for name in ["fold1_candidate_eval", "fold1_policy_scores",
             "gold_map", "champion_model_gold_audit"]:
    obj = globals().get(name)
    print(f"\n{name}:")
    print(repr(obj)[:5000])

print("\n=== RELEVANT FUNCTION DEFINITIONS ===")
for name in ["eval_variants", "counsel_f1", "counsel_spans",
             "predict_texts", "merge_policy"]:
    obj = globals().get(name)
    print(f"\n--- {name} ---")
    if callable(obj):
        try:
            print(inspect.getsource(obj)[:6000])
        except Exception as e:
            print("Source unavailable:", repr(e))
    else:
        print("Not defined")

print("\n=== AVAILABLE TEXT/GOLD KEYS ===")
for name in ["train_texts", "train_gold", "val_texts", "val_gold",
             "texts", "gold"]:
    obj = globals().get(name)
    if isinstance(obj, dict):
        print(name, "documents:", len(obj),
              "sample IDs:", list(obj.keys())[:5])


In [ ]:

# CS781 — Verify fold-1 split and evaluate counsel V5
# Does not modify model weights or write prediction files.

import pandas as pd

assert len(train_texts) == len(train_gold) == 35
assert set(train_texts) == set(train_gold)

# Check whether fold1_candidate_eval's original spans correspond to
# annotations in train_gold or val_gold.
candidate_eval = globals().get("fold1_candidate_eval", {})
candidate_gold = candidate_eval.get("gold", set())
candidate_original = candidate_eval.get("original", set())

def span_keys(gold_dict):
    return {
        (str(doc_id), int(ent[0]), int(ent[1]))
        for doc_id, ents in gold_dict.items()
        for ent in ents
        if len(ent) >= 3
    }

train_keys = span_keys(train_gold)
val_keys = span_keys(val_gold)

print("=== SPLIT IDENTITY CHECK ===")
print("train docs:", len(train_texts), "val docs:", len(val_texts))
print("train/val document overlap:", len(set(train_texts) & set(val_texts)))
print("fold1 candidate gold spans:", len(candidate_gold))
print("overlap with train gold:", len(candidate_gold & train_keys))
print("overlap with val gold:", len(candidate_gold & val_keys))
print("fold1 original spans:", len(candidate_original))

# Refuse to call this fold-1 validation unless the metadata supports it.
if len(candidate_gold) and len(candidate_gold & train_keys) > len(candidate_gold & val_keys):
    eval_texts, eval_gold = train_texts, train_gold
    split_name = "train_texts/train_gold (matches fold1 candidate metadata)"
elif len(candidate_gold) and len(candidate_gold & val_keys) > len(candidate_gold & train_keys):
    eval_texts, eval_gold = val_texts, val_gold
    split_name = "val_texts/val_gold (matches fold1 candidate metadata)"
else:
    raise RuntimeError(
        "The fold-1 metadata does not uniquely identify train or validation. "
        "No score was computed to avoid a misleading result."
    )

print("\nEvaluation split:", split_name)

def exact_entity_score(predictions, gold):
    labels = sorted({
        str(e[2])
        for ents in gold.values()
        for e in ents if len(e) >= 3
    })
    gs = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    ps = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in predictions.items()
        for e in ents if len(e) >= 3
    }
    rows = []
    for lab in labels:
        g = {x for x in gs if x[3] == lab}
        p = {x for x in ps if x[3] == lab}
        tp = len(g & p)
        pr = tp / len(p) if p else 0.0
        rc = tp / len(g) if g else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        rows.append({
            "label": lab, "gold": len(g), "predicted": len(p),
            "TP": tp, "precision": pr, "recall": rc, "F1": f1
        })
    report = pd.DataFrame(rows).set_index("label")
    return float(report.F1.mean()), report

# Generate baseline predictions with the existing champion.
base_raw = predict_texts(champion_model, eval_texts)
base_filtered = apply_thr(base_raw, champ_thr)

# Correct argument order, matching the notebook's function definition.
v5_predictions = merge_policy(eval_texts, base_filtered, "drop")

base_f1, base_report = exact_entity_score(base_filtered, eval_gold)
v5_f1, v5_report = exact_entity_score(v5_predictions, eval_gold)

print("\n=== ENTITY-LEVEL RESULTS ===")
print("Split:", split_name)
print(f"Champion:       {base_f1:.4f}")
print(f"Champion + V5:  {v5_f1:.4f}")
print(f"Delta:          {v5_f1 - base_f1:+.4f}")

print("\n=== PER-LABEL COMPARISON ===")
comparison = pd.DataFrame({
    "champion_F1": base_report.F1,
    "V5_F1": v5_report.F1,
    "champion_pred": base_report.predicted,
    "V5_pred": v5_report.predicted,
})
print(comparison.round(3).to_string())

print("\nNo files or model weights were modified.")


In [ ]:

# CS781 — Held-out validation evaluation + counsel error audit
# Read-only with respect to model checkpoints and prediction files.

import pandas as pd

assert len(val_texts) == len(val_gold) == 35
assert set(val_texts).isdisjoint(set(train_texts)), (
    "Train and validation document IDs overlap; verify split provenance."
)

def exact_span_report(predictions, gold):
    labels = sorted({
        str(e[2])
        for ents in gold.values()
        for e in ents if len(e) >= 3
    })

    gs = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    ps = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gs if x[3] == label}
        p = {x for x in ps if x[3] == label}
        tp = len(g & p)
        fp, fn = len(p - g), len(g - p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows.append({
            "label": label, "gold": len(g), "predicted": len(p),
            "TP": tp, "FP": fp, "FN": fn,
            "precision": precision, "recall": recall, "F1": f1
        })

    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report, gs, ps

# Recompute on validation documents; don't reuse train-side predictions.
val_base = apply_thr(predict_texts(champion_model, val_texts), champ_thr)
val_v5 = merge_policy(val_texts, val_base, "drop")

base_macro, base_report, gold_spans, base_spans = exact_span_report(
    val_base, val_gold
)
v5_macro, v5_report, _, v5_spans = exact_span_report(val_v5, val_gold)

print("=== VALIDATION RESULTS ===")
print(f"Champion:       {base_macro:.4f}")
print(f"Champion + V5:  {v5_macro:.4f}")
print(f"Delta:          {v5_macro - base_macro:+.4f}")

print("\n=== PER-LABEL COMPARISON ===")
display(pd.DataFrame({
    "champion_F1": base_report["F1"],
    "V5_F1": v5_report["F1"],
    "champion_pred": base_report["predicted"],
    "V5_pred": v5_report["predicted"],
}).round(3))

# Counsel-only exact-match audit.
counsel_labels = {"A.COUNSEL", "R.COUNSEL"}
c_gold = {x for x in gold_spans if x[3] in counsel_labels}
c_base = {x for x in base_spans if x[3] in counsel_labels}
c_v5 = {x for x in v5_spans if x[3] in counsel_labels}

def counsel_summary(gold, pred, title):
    tp = gold & pred
    fp = pred - gold
    fn = gold - pred
    p = len(tp) / len(pred) if pred else 0.0
    r = len(tp) / len(gold) if gold else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    print(f"\n{title}: TP={len(tp)} FP={len(fp)} FN={len(fn)} "
          f"P={p:.3f} R={r:.3f} F1={f:.3f}")
    return fp, fn

base_fp, base_fn = counsel_summary(c_gold, c_base, "Champion counsel")
v5_fp, v5_fn = counsel_summary(c_gold, c_v5, "Champion + V5 counsel")

# Print text snippets for V5 false positives and missed gold counsel spans.
def show_examples(spans, texts, heading, limit=15):
    print(f"\n=== {heading} (up to {limit}) ===")
    for doc, start, end, label in sorted(spans)[:limit]:
        text = texts.get(doc)
        if text is None:
            print((doc, start, end, label), "[document text unavailable]")
            continue
        lo, hi = max(0, start - 90), min(len(text), end + 90)
        marked = text[lo:start] + " ⟦" + text[start:end] + "⟧ " + text[end:hi]
        print(f"{label} | doc={doc} | span=({start},{end}) | {marked!r}")

show_examples(v5_fp, val_texts, "V5 COUNSEL FALSE POSITIVES")
show_examples(v5_fn, val_texts, "V5 COUNSEL MISSED GOLD SPANS")

print("\nNo model weights or prediction files were modified.")


In [ ]:

# CS781 — Inspect training/split provenance without changing anything

import inspect

print("=== MODEL OBJECT ===")
print("champion_model type:", type(champion_model))
print("model class:", champion_model.__class__.__name__)

print("\n=== POSSIBLE TRAINING METADATA ===")
keywords = (
    "train", "fold", "split", "dataset", "checkpoint",
    "best_epoch", "history", "trainer", "config"
)

for name, obj in list(globals().items()):
    if not any(k in name.lower() for k in keywords):
        continue
    if name.startswith("_"):
        continue
    if isinstance(obj, (dict, list, tuple, str, int, float, bool)):
        try:
            summary = repr(obj)
            if len(summary) > 1000:
                summary = summary[:1000] + "... [truncated]"
            print(f"\n{name} ({type(obj).__name__}): {summary}")
        except Exception:
            pass

print("\n=== AVAILABLE CHECKPOINT PATHS ===")
for name in ["CHAMPION_PATH", "MODEL_PATH", "BEST_MODEL_PATH",
             "champion_path", "checkpoint_path", "best_path"]:
    if name in globals():
        print(name, "=", repr(globals()[name]))

print("\nInspect the original training cell/config to establish whether")
print("the champion was trained on train_texts, val_texts, or another split.")
print("No weights or files were modified.")


In [ ]:

# CS781 — V5 counsel error breakdown by candidate source
# No retraining, checkpoint changes, or file writes.

from collections import defaultdict
import pandas as pd

assert "val_texts" in globals() and "val_gold" in globals()
assert "rule_spans" in globals()
assert "cleaned_counsel_candidates" in globals()
assert "explicit_side" in globals()

LABELS = {"A.COUNSEL", "R.COUNSEL"}

gold_labeled = set()
original_labeled = set()
cleaned_labeled = set()

for doc_id, entities in val_gold.items():
    for ent in entities:
        if len(ent) >= 3 and ent[2] in LABELS:
            gold_labeled.add(
                (str(doc_id), int(ent[0]), int(ent[1]), str(ent[2]))
            )

for doc_id, text in val_texts.items():
    for s, e in rule_spans(text):
        label = explicit_side(text, s, e)
        if label in LABELS:
            original_labeled.add(
                (str(doc_id), int(s), int(e), label)
            )

    for s, e in cleaned_counsel_candidates(text):
        label = explicit_side(text, s, e)
        if label in LABELS:
            cleaned_labeled.add(
                (str(doc_id), int(s), int(e), label)
            )

def metric_row(name, pred):
    tp = len(pred & gold_labeled)
    fp = len(pred - gold_labeled)
    fn = len(gold_labeled - pred)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold_labeled) if gold_labeled else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return {
        "candidate_source": name,
        "predicted": len(pred),
        "TP": tp, "FP": fp, "FN": fn,
        "precision": p, "recall": r, "F1": f
    }

combined = original_labeled | cleaned_labeled
results = pd.DataFrame([
    metric_row("Original rules", original_labeled),
    metric_row("Cleaned candidates", cleaned_labeled),
    metric_row("Union", combined),
])
display(results.round(3))

# Display errors from the union for manual review.
false_positives = sorted(combined - gold_labeled)
false_negatives = sorted(gold_labeled - combined)

def print_spans(spans, heading, limit=25):
    print(f"\n=== {heading}: {len(spans)} total ===")
    for doc_id, s, e, label in spans[:limit]:
        text = val_texts.get(doc_id)
        if text is None:
            text = val_texts.get(next(
                (k for k in val_texts if str(k) == doc_id), None
            ))
        if text is None:
            print(doc_id, s, e, label, "[text missing]")
            continue

        print(
            f"{label} | {text[max(0,s-100):s]!r}"
            f" ⟦{text[s:e]}⟧ "
            f"{text[e:min(len(text),e+100)]!r}"
        )

print_spans(false_positives, "V5 RULE FALSE POSITIVES")
print_spans(false_negatives, "V5 RULE MISSED GOLD")

print("\nThis cell measures rule candidates separately from the full model.")
print("Do not replace the existing V5 pipeline based on these metrics alone.")


In [ ]:

# CS781 — Conservative counsel-rule experiment
# Only evaluates alternatives; does not modify the champion or save files.

import re
import pandas as pd

COUNSEL_ROLE = re.compile(
    r"\b(?:"
    r"learned\s+(?:senior\s+)?counsel|"
    r"senior\s+advocate|"
    r"advocate\s+for|"
    r"counsel\s+for|"
    r"appearing\s+for|"
    r"on\s+behalf\s+of|"
    r"for\s+the\s+(?:appellant|appellants|petitioner|petitioners|"
    r"respondent|respondents|state|revenue|prosecution|complainant)"
    r")\b",
    re.I
)

NEGATIVE_ROLE = re.compile(
    r"\b(?:"
    r"appellant(?:\s+no\.?\s*\d+)?|"
    r"petitioner\s+in\s+person|"
    r"appeared\s+in\s+person|"
    r"witness|deponent|informant|"
    r"investigating\s+officer|"
    r"amicus\s+curiae"
    r")\b",
    re.I
)

def has_nearby_counsel_role(text, start, end, window=150):
    left = text[max(0, start-window):start]
    right = text[end:min(len(text), end+window)]

    # Strong legal-role cues immediately near the candidate.
    if COUNSEL_ROLE.search(left) or COUNSEL_ROLE.search(right):
        # Avoid treating a nearby explicit non-counsel role as sufficient.
        nearby = left[-100:] + " " + right[:100]
        if NEGATIVE_ROLE.search(nearby) and not re.search(
            r"\b(?:senior\s+advocate|learned\s+counsel|counsel\s+for|"
            r"advocate\s+for|appearing\s+for)\b",
            nearby, re.I
        ):
            return False
        return True
    return False

def build_rule_predictions(texts, require_cue=False, window=150):
    out = {}
    for doc_id, text in texts.items():
        spans = set()

        # Use the existing candidate sources.
        candidates = list(rule_spans(text))
        candidates += list(cleaned_counsel_candidates(text))

        for start, end in candidates:
            if start < 0 or end <= start or end > len(text):
                continue

            label = explicit_side(text, start, end)
            if label not in {"A.COUNSEL", "R.COUNSEL"}:
                continue

            if require_cue and not has_nearby_counsel_role(
                text, start, end, window=window
            ):
                continue

            spans.add((str(doc_id), int(start), int(end), label))

        out[str(doc_id)] = [
            (s, e, label)
            for d, s, e, label in sorted(spans)
        ]

    return out

# Gold counsel spans in validation.
gold_counsel = {
    (str(doc), int(e[0]), int(e[1]), str(e[2]))
    for doc, ents in val_gold.items()
    for e in ents
    if len(e) >= 3 and e[2] in {"A.COUNSEL", "R.COUNSEL"}
}

def flatten_predictions(pred):
    return {
        (str(doc), int(e[0]), int(e[1]), str(e[2]))
        for doc, ents in pred.items()
        for e in ents if len(e) >= 3
        and e[2] in {"A.COUNSEL", "R.COUNSEL"}
    }

def counsel_metrics(name, pred_set):
    tp = len(pred_set & gold_counsel)
    fp = len(pred_set - gold_counsel)
    fn = len(gold_counsel - pred_set)
    p = tp / len(pred_set) if pred_set else 0.0
    r = tp / len(gold_counsel) if gold_counsel else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return {
        "variant": name, "pred": len(pred_set),
        "TP": tp, "FP": fp, "FN": fn,
        "precision": p, "recall": r, "F1": f
    }

variants = []
for window in [50, 100, 150]:
    pred = build_rule_predictions(
        val_texts, require_cue=True, window=window
    )
    variants.append(
        counsel_metrics(f"role cue ±{window} chars", flatten_predictions(pred))
    )

# Reference: current full V5 counsel predictions.
current_counsel = {
    (str(doc), int(e[0]), int(e[1]), str(e[2]))
    for doc, ents in val_v5.items()
    for e in ents if len(e) >= 3
    and e[2] in {"A.COUNSEL", "R.COUNSEL"}
}
variants.insert(0, counsel_metrics("current V5", current_counsel))

print("=== COUNSEL FILTER COMPARISON ===")
display(pd.DataFrame(variants).set_index("variant").round(3))

print(
    "\nInterpretation: a stricter filter is useful only if it reduces false "
    "positives without collapsing recall. This is a validation experiment; "
    "keep the original V5 output unchanged."
)


In [ ]:

# CS781 — Compare counsel side-assignment policies on validation
# Read-only experiment.

import pandas as pd

assert "val_texts" in globals() and "val_gold" in globals()
assert "val_base" in globals()
assert "merge_policy" in globals()

def entity_report(predictions, gold):
    labels = sorted({
        str(e[2]) for ents in gold.values()
        for e in ents if len(e) >= 3
    })
    gs = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    ps = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for lab in labels:
        g = {x for x in gs if x[3] == lab}
        p = {x for x in ps if x[3] == lab}
        tp = len(g & p)
        pr = tp / len(p) if p else 0.0
        rc = tp / len(g) if g else 0.0
        f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
        rows.append({
            "label": lab, "F1": f1,
            "gold": len(g), "predicted": len(p),
            "TP": tp, "FP": len(p-g), "FN": len(g-p)
        })
    return float(pd.DataFrame(rows)["F1"].mean()), pd.DataFrame(rows).set_index("label")

policy_results = []
policy_predictions = {}

for policy in ["R", "A", "drop"]:
    pred = merge_policy(val_texts, val_base, policy)
    macro, report = entity_report(pred, val_gold)
    policy_predictions[policy] = pred

    counsel_gold = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in val_gold.items()
        for e in ents if len(e) >= 3
        and e[2] in {"A.COUNSEL", "R.COUNSEL"}
    }
    counsel_pred = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in pred.items()
        for e in ents if len(e) >= 3
        and e[2] in {"A.COUNSEL", "R.COUNSEL"}
    }
    tp = len(counsel_gold & counsel_pred)
    p = tp / len(counsel_pred) if counsel_pred else 0
    r = tp / len(counsel_gold) if counsel_gold else 0
    f = 2*p*r/(p+r) if p+r else 0

    policy_results.append({
        "policy": policy,
        "macro_F1": macro,
        "counsel_precision": p,
        "counsel_recall": r,
        "counsel_F1": f,
        "counsel_predictions": len(counsel_pred),
    })

policy_table = pd.DataFrame(policy_results).set_index("policy")
display(policy_table.round(4))

print("Best policy by full macro-F1:", policy_table["macro_F1"].idxmax())
print("This is an evaluation only; no model or files were changed.")


In [ ]:

# CS781 — Per-label comparison of counsel policies
# Read-only analysis.

import pandas as pd

def exact_counts(predictions, gold):
    labels = sorted({
        str(e[2])
        for ents in gold.values()
        for e in ents if len(e) >= 3
    })
    gs = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for policy, pred in predictions.items():
        ps = {
            (str(d), int(e[0]), int(e[1]), str(e[2]))
            for d, ents in pred.items()
            for e in ents if len(e) >= 3
        }
        for label in labels:
            g = {x for x in gs if x[3] == label}
            p = {x for x in ps if x[3] == label}
            tp = len(g & p)
            precision = tp / len(p) if p else 0.0
            recall = tp / len(g) if g else 0.0
            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall else 0.0
            )
            rows.append({
                "policy": policy,
                "label": label,
                "F1": f1,
                "predicted": len(p),
                "TP": tp,
                "FP": len(p-g),
                "FN": len(g-p),
            })
    return pd.DataFrame(rows)

per_label = exact_counts(policy_predictions, val_gold)

wide = per_label.pivot(index="label", columns="policy", values="F1")
wide["R_minus_drop"] = wide["R"] - wide["drop"]
print("=== PER-LABEL F1 ===")
display(wide.round(4))

counts = per_label[per_label["label"].isin(["A.COUNSEL", "R.COUNSEL"])]
print("\n=== COUNSEL COUNTS ===")
display(counts.sort_values(["label", "policy"]).round(4))

print("\nNo model weights or prediction files were modified.")


In [ ]:

# CS781 — Hybrid counsel policy: A.COUNSEL from policy A,
# R.COUNSEL from policy R. All other predictions stay unchanged.
# Evaluation only: no model/checkpoint/file modifications.

import pandas as pd

assert all(p in policy_predictions for p in ["A", "R", "drop"])

def make_hybrid(base, pred_A, pred_R):
    result = {}
    for doc_id, ents in base.items():
        kept = [
            tuple(e) for e in ents
            if len(e) >= 3 and e[2] not in {"A.COUNSEL", "R.COUNSEL"}
        ]
        a_ents = [
            tuple(e) for e in pred_A.get(doc_id, [])
            if len(e) >= 3 and e[2] == "A.COUNSEL"
        ]
        r_ents = [
            tuple(e) for e in pred_R.get(doc_id, [])
            if len(e) >= 3 and e[2] == "R.COUNSEL"
        ]
        result[doc_id] = kept + a_ents + r_ents
    return result

hybrid_pred = make_hybrid(
    val_base,
    policy_predictions["A"],
    policy_predictions["R"]
)

def exact_macro_and_report(predictions, gold):
    labels = sorted({
        str(e[2]) for ents in gold.values()
        for e in ents if len(e) >= 3
    })
    gs = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    ps = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for lab in labels:
        g = {x for x in gs if x[3] == lab}
        p = {x for x in ps if x[3] == lab}
        tp = len(g & p)
        pr = tp / len(p) if p else 0.0
        rc = tp / len(g) if g else 0.0
        f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
        rows.append({
            "label": lab, "F1": f1,
            "predicted": len(p), "TP": tp,
            "FP": len(p-g), "FN": len(g-p)
        })
    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report

hybrid_macro, hybrid_report = exact_macro_and_report(hybrid_pred, val_gold)

summary = pd.DataFrame({
    "policy_R": exact_macro_and_report(policy_predictions["R"], val_gold)[1]["F1"],
    "policy_A": exact_macro_and_report(policy_predictions["A"], val_gold)[1]["F1"],
    "policy_drop": exact_macro_and_report(policy_predictions["drop"], val_gold)[1]["F1"],
    "hybrid_A_for_A_R_for_R": hybrid_report["F1"],
})
print("=== FULL MACRO-F1 ===")
print(f"R:      {exact_macro_and_report(policy_predictions['R'], val_gold)[0]:.4f}")
print(f"A:      {exact_macro_and_report(policy_predictions['A'], val_gold)[0]:.4f}")
print(f"drop:   {exact_macro_and_report(policy_predictions['drop'], val_gold)[0]:.4f}")
print(f"hybrid: {hybrid_macro:.4f}")

print("\n=== PER-LABEL F1 ===")
display(summary.round(4))

print("\n=== HYBRID COUNSEL COUNTS ===")
display(hybrid_report.loc[["A.COUNSEL", "R.COUNSEL"]].round(4))

print("\nNo model weights or prediction files were modified.")


In [ ]:

# CS781 — Counsel confidence sweep on held-out validation
# Read-only; keeps the champion and current predictions unchanged.

import pandas as pd

assert "raw_fold2" in globals(), "raw_fold2 not found."
assert "val_gold" in globals(), "val_gold not found."

sample_doc = next(iter(raw_fold2))
print("Sample raw prediction:", raw_fold2[sample_doc][:5])

# raw_fold2 entries are expected to be (start, end, label, confidence).
# Stop rather than silently misinterpret a different format.
sample = next(
    (e for ents in raw_fold2.values() for e in ents if len(e) >= 4),
    None
)
if sample is None or not isinstance(sample[3], (int, float)):
    raise RuntimeError(
        "raw_fold2 does not appear to contain numeric confidence scores "
        "in position 4. No threshold experiment was run."
    )

gold_counsel = {
    (str(d), int(e[0]), int(e[1]), str(e[2]))
    for d, ents in val_gold.items()
    for e in ents if len(e) >= 3
    and e[2] in {"A.COUNSEL", "R.COUNSEL"}
}

# Keep only rule-generated counsel spans whose confidence can be matched
# to an exact raw model span. This tests model confidence, not rule quality.
def counsel_confidence_sweep(raw, gold):
    raw_lookup = {}
    for doc, ents in raw.items():
        for e in ents:
            if len(e) >= 4:
                s, end, label, conf = e[:4]
                if label in {"A.COUNSEL", "R.COUNSEL"}:
                    raw_lookup[(str(doc), int(s), int(end), str(label))] = float(conf)

    rows = []
    for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
        pred = {
            span for span, conf in raw_lookup.items()
            if conf >= threshold
        }
        tp = len(pred & gold)
        fp = len(pred - gold)
        fn = len(gold - pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold) if gold else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        rows.append({
            "confidence_threshold": threshold,
            "predicted": len(pred), "TP": tp, "FP": fp, "FN": fn,
            "precision": p, "recall": r, "F1": f1
        })
    return pd.DataFrame(rows)

display(counsel_confidence_sweep(raw_fold2, gold_counsel).round(3))
print(
    "\nNote: this sweep evaluates only model-generated counsel spans with "
    "matching exact boundaries. It does not replace the V5 rule evaluation."
)


In [ ]:

# CS781 — Freeze experimental policy R and run consistency checks.
# No file writes; no model weight changes.

import pandas as pd

assert "policy_predictions" in globals()
assert "R" in policy_predictions
assert "val_texts" in globals() and "val_gold" in globals()

# Preserve a separate in-memory copy; do not overwrite val_v5 or champion outputs.
experimental_policy = "R"
experimental_val_predictions = {
    doc_id: [tuple(e) for e in ents]
    for doc_id, ents in policy_predictions["R"].items()
}

assert set(experimental_val_predictions) == set(val_gold)

# Validate spans and labels.
allowed_labels = {
    str(e[2])
    for ents in val_gold.values()
    for e in ents if len(e) >= 3
}
invalid = []
for doc_id, ents in experimental_val_predictions.items():
    text = val_texts[doc_id]
    for ent in ents:
        if len(ent) < 3:
            invalid.append((doc_id, ent, "short tuple"))
            continue
        s, end, label = int(ent[0]), int(ent[1]), str(ent[2])
        if s < 0 or end <= s or end > len(text):
            invalid.append((doc_id, ent, "invalid boundaries"))
        if label not in allowed_labels:
            invalid.append((doc_id, ent, "unknown label"))

if invalid:
    print("Invalid spans found:", len(invalid))
    print(invalid[:20])
else:
    print("All predicted spans have valid boundaries and known labels.")

counts = {}
for doc_id, ents in experimental_val_predictions.items():
    for ent in ents:
        label = str(ent[2])
        counts[label] = counts.get(label, 0) + 1

print("\nExperimental policy:", experimental_policy)
print("Documents:", len(experimental_val_predictions))
print("Total predicted spans:", sum(counts.values()))
print("\nEntity counts:")
display(
    pd.DataFrame(
        sorted(counts.items()),
        columns=["label", "predictions"]
    ).set_index("label")
)

print("\nExperimental predictions remain in memory only.")
print("Champion model and saved prediction files are untouched.")


In [ ]:

# CS781 — Final validation report: Champion vs experimental policy R
# Read-only. Uses existing predictions and exact entity-span matching.

import pandas as pd

def final_exact_span_report(predictions, gold):
    labels = sorted({
        str(e[2])
        for ents in gold.values()
        for e in ents if len(e) >= 3
    })

    gold_set = {
        (str(doc), int(e[0]), int(e[1]), str(e[2]))
        for doc, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    pred_set = {
        (str(doc), int(e[0]), int(e[1]), str(e[2]))
        for doc, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gold_set if x[3] == label}
        p = {x for x in pred_set if x[3] == label}
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows.append({
            "label": label,
            "gold": len(g),
            "predicted": len(p),
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "F1": f1,
        })

    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report

# Reuse the baseline that was generated on val_texts.
assert "val_base" in globals()
assert "experimental_val_predictions" in globals()

champion_macro, champion_report = final_exact_span_report(val_base, val_gold)
policy_r_macro, policy_r_report = final_exact_span_report(
    experimental_val_predictions, val_gold
)

print("=== FINAL HELD-OUT VALIDATION COMPARISON ===")
print(f"Champion:  {champion_macro:.4f}")
print(f"Policy R:  {policy_r_macro:.4f}")
print(f"Delta:     {policy_r_macro - champion_macro:+.4f}")

print("\n=== PER-ENTITY F1 ===")
display(pd.DataFrame({
    "champion_F1": champion_report["F1"],
    "policy_R_F1": policy_r_report["F1"],
    "champion_pred": champion_report["predicted"],
    "policy_R_pred": policy_r_report["predicted"],
}).round(4))

print("\n=== MACRO-F1 CHECK ===")
print("Macro-F1 is the unweighted mean across entity labels.")
print("These results are validation metrics, not final test-set metrics.")
print("No model weights or prediction files were modified.")


In [ ]:
# ============================================================
# FIXED TEST INFERENCE — POLICY R
# Both inputs to merge_policy are dictionaries.
# ============================================================

import json
from collections import Counter

# 1. Check test data structure
print("test_texts type:", type(test_texts).__name__)
print("test_texts length:", len(test_texts))

# merge_policy expects texts.items(), so construct a dictionary.
if isinstance(test_texts, dict):
    test_texts_by_id = test_texts
else:
    # If test_texts is a list, use the IDs returned by predict_texts
    # to align the texts with the predictions.
    test_texts_by_id = None

# 2. Run champion inference
test_raw_predictions = predict_texts(champion_model, test_texts)
test_base_predictions = apply_thr(test_raw_predictions, champ_thr)

if not isinstance(test_base_predictions, dict):
    raise TypeError(
        f"apply_thr returned {type(test_base_predictions).__name__}, "
        "but a dictionary is required."
    )

test_ids = list(test_base_predictions.keys())

# Align document text IDs to prediction IDs.
if test_texts_by_id is None:
    if len(test_ids) != len(test_texts):
        raise ValueError(
            f"Cannot align documents: {len(test_ids)} prediction IDs "
            f"but {len(test_texts)} texts."
        )
    test_texts_by_id = dict(zip(test_ids, test_texts))

if set(test_texts_by_id.keys()) != set(test_base_predictions.keys()):
    raise ValueError(
        "Text and prediction document IDs do not match. "
        "Do not generate output until the correct test IDs are aligned."
    )

print("Aligned test documents:", len(test_ids))

# 3. Apply policy R using dictionaries, as required by merge_policy()
test_policy_predictions = merge_policy(
    test_texts_by_id,
    test_base_predictions,
    "R"
)

if not isinstance(test_policy_predictions, dict):
    raise TypeError(
        "Expected merge_policy to return a dictionary, got "
        f"{type(test_policy_predictions).__name__}."
    )

# 4. Validate predictions
allowed_labels = {
    "A.COUNSEL", "R.COUNSEL", "APP", "RESP", "JUDGE",
    "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
}

label_counts = Counter()
total_spans = 0

for doc_id, spans in test_policy_predictions.items():
    text = test_texts_by_id[doc_id]

    for span in spans:
        start, end, label = span[:3]

        if not isinstance(start, int) or not isinstance(end, int):
            raise TypeError(f"Invalid offsets: document={doc_id}, span={span}")

        if not (0 <= start < end <= len(text)):
            raise ValueError(
                f"Invalid span: document={doc_id}, span={span}, "
                f"text length={len(text)}"
            )

        if label not in allowed_labels:
            raise ValueError(f"Unexpected label: {label}")

        label_counts[label] += 1
        total_spans += 1

print("Validated spans:", total_spans)
print("Label counts:", dict(sorted(label_counts.items())))

# 5. Write a separate intermediate JSON
# Format: {document_id: [[start, end, label], ...]}
prediction_dict = {
    str(doc_id): [
        [int(span[0]), int(span[1]), str(span[2])]
        for span in sorted(spans, key=lambda s: (s[0], s[1], s[2]))
    ]
    for doc_id, spans in test_policy_predictions.items()
}

output_path = "/kaggle/working/prediction.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(prediction_dict, f, ensure_ascii=False, indent=2)

# 6. Verify saved file
with open(output_path, "r", encoding="utf-8") as f:
    check = json.load(f)

assert len(check) == len(test_texts_by_id)
assert sum(len(v) for v in check.values()) == total_spans

print("\nSUCCESS")
print("Saved:", output_path)
print("Documents:", len(check))
print("Total spans:", total_spans)
print("Preview:", next(iter(check.items())))

In [ ]:
import random
from collections import defaultdict
random.seed(1)

HON_BEFORE = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")

def side_state(text, s):
    ctx = text[max(0, s - 600):s]
    a = [m.start() for m in SIDE_A.finditer(ctx)]; r = [m.start() for m in SIDE_R.finditer(ctx)]
    if not a and not r: return "none"
    if a and not r: return "A-cue only"
    if r and not a: return "R-cue only"
    return "both (nearer wins)"

def side_error_report(texts, gold, name, show=6):
    acc = defaultdict(lambda: [0, 0]); wrong, missed = [], []
    for d, text in texts.items():
        gl = {(int(e[0]), int(e[1])): e[2] for e in gold[d] if e[2] in ("A.COUNSEL", "R.COUNSEL")}
        rule = set(rule_spans(text))
        for (s, e) in rule:
            if (s, e) in gl:
                st, pred = side_state(text, s), side(text, s)
                ok = pred == gl[(s, e)]
                acc[st][0] += ok; acc[st][1] += 1
                if not ok:
                    wrong.append((text[max(0, s - 160):s].replace("\n", " "), text[s:e], gl[(s, e)], pred, st))
        for (s, e), lab in gl.items():
            if (s, e) not in rule:
                left = text[max(0, s - 40):s]
                missed.append((left.replace("\n", " "), text[s:e].replace("\n", " "), lab, bool(HON_BEFORE.search(left))))
    print(f"\n=== {name}: side accuracy on boundary-correct spans, by cue state ===")
    for st, (ok, n) in sorted(acc.items()):
        print(f"  {st:20s} {ok:3d}/{n:3d} = {ok/max(n,1):.2f}")
    print(f"\n--- {min(show, len(wrong))} wrong-side examples (160 chars of left context) ---")
    for ctx, nm, gl_lab, pr, st in random.sample(wrong, min(show, len(wrong))):
        print(f"[{st}] gold={gl_lab} pred={pr}\n   ctx: ...{ctx!r}\n   name: {nm!r}")
    n_hon = sum(1 for m in missed if m[3])
    print(f"\n--- missed gold counsel spans: {len(missed)} total; {n_hon} follow an honorific (boundary issue), "
          f"{len(missed) - n_hon} do not (no honorific) ---")
    for left, nm, lab, hon in random.sample(missed, min(show, len(missed))):
        print(f"gold={lab} honorific_before={hon}\n   left: ...{left!r}\n   gold: {nm!r}")

side_error_report(train_texts, train_gold, "FOLD 1 (dev)")

In [ ]:
PARTY_A = r"(?:petitioners?|appellants?|applicants?|plaintiffs?|complainants?|prosecution|revisionists?)"
PARTY_R = r"(?:respondents?|defendants?|accused|non-\s*applicants?|opposite\s+part(?:y|ies))"
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"
CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s\.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s\.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")
A_, R_ = "A.COUNSEL", "R.COUNSEL"

def _cues(text, lo, hi):
    out = [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]
    out += [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]
    return out

def assign_side(text, s, e, order, prev, W_L=300, W_R=200):
    for step in order:
        if step == "right":
            c = _cues(text, e, min(len(text), e + W_R))
            if c: return min(c)[2], "right"
        elif step == "left":
            c = _cues(text, max(0, s - W_L), s)
            c = [x for x in c if x[1] <= s]
            if c: return max(c, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail): return A_, "hdr"
            if HDR_R.search(tail): return R_, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_, "default"

VARIANTS = {"V1 left-for": ["left"], "V2 right-for": ["right"], "V3 right>left": ["right", "left"],
            "V4 left>right": ["left", "right"], "V5 hdr>right>left>inh": ["hdr", "right", "left", "inherit"],
            "V6 right>hdr>left>inh": ["right", "hdr", "left", "inherit"]}

def eval_variants(texts, gold):
    G = {A_: set(), R_: set()}
    for d, ents in gold.items():
        for x in ents:
            if x[2] in G: G[x[2]].add((d, int(x[0]), int(x[1])))
    gl = {}
    for lab, st in G.items():
        for k in st: gl[k] = lab
    rows, src_stats = [], None
    runs = {"V0 current (600-char any-word)": None, **VARIANTS}
    for name, order in runs.items():
        P = {A_: set(), R_: set()}; ok = n = cue = tot = 0; stats = {}
        for d, text in texts.items():
            prev = None
            for s, e in sorted(rule_spans(text)):
                if order is None:
                    lab, src = side(text, s), ("cue" if side_state(text, s) != "none" else "default")
                else:
                    lab, src = assign_side(text, s, e, order, prev)
                    prev = (e, lab, src)
                P[lab].add((d, s, e)); tot += 1; cue += (src != "default")
                if (d, s, e) in gl:
                    n += 1; hit = (lab == gl[(d, s, e)]); ok += hit
                    t = stats.setdefault(src, [0, 0]); t[0] += hit; t[1] += 1
        f = {}
        for lab in (A_, R_):
            tp = len(G[lab] & P[lab]); p = tp / max(len(P[lab]), 1); r = tp / max(len(G[lab]), 1)
            f[lab] = 2 * p * r / (p + r) if p + r else 0.0
        rows.append({"variant": name, "side_acc": ok / n, "n_bnd_correct": n, "cue_coverage": cue / tot,
                     "A.COUNSEL_F1": f[A_], "R.COUNSEL_F1": f[R_], "A+R_mean": (f[A_] + f[R_]) / 2})
        if name.startswith("V5"): src_stats = stats
    print(pd.DataFrame(rows).round(3).to_string(index=False))
    print("\nV5 accuracy by cue source (boundary-correct spans):")
    for src, (h, k) in src_stats.items(): print(f"  {src:8s} {h:3d}/{k:3d} = {h/k:.2f}")

eval_variants(train_texts, train_gold)   # FOLD 1 ONLY

In [ ]:
ORDER = ["hdr", "right", "left", "inherit"]   # V5

def counsel_spans(text, policy):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        lab, src = assign_side(text, s, e, ORDER, prev)
        prev = (e, lab, src)
        if src == "default":
            if policy == "drop": continue
            if policy == "A": lab = A_
        out.append((s, e, lab))
    return out

def counsel_f1(texts, gold, policy):
    G = {A_: set(), R_: set()}; P = {A_: set(), R_: set()}
    for d, ents in gold.items():
        for x in ents:
            if x[2] in G: G[x[2]].add((d, int(x[0]), int(x[1])))
    for d, text in texts.items():
        for s, e, lab in counsel_spans(text, policy): P[lab].add((d, s, e))
    f = {}
    for lab in (A_, R_):
        tp = len(G[lab] & P[lab]); p = tp / max(len(P[lab]), 1); r = tp / max(len(G[lab]), 1)
        f[lab] = 2 * p * r / (p + r) if p + r else 0.0
    return f

# ---- 1. choose default policy on FOLD 1 only ----
fold1 = {pol: counsel_f1(train_texts, train_gold, pol) for pol in ["R", "A", "drop"]}
for pol, f in fold1.items():
    print(f"fold 1, default={pol:5s}: A={f[A_]:.3f} R={f[R_]:.3f} mean={(f[A_]+f[R_])/2:.3f}")
chosen = max(fold1, key=lambda p: (fold1[p][A_] + fold1[p][R_], p == "R"))
print("chosen default policy (fold 1):", chosen)

# ---- 2. fold 2, once ----
def predict_texts(model, texts):
    preds = {}
    for doc_id, text in texts.items():
        offsets, probs = diag_doc_probs(model, text)
        if len(offsets) == 0: preds[doc_id] = []; continue
        pt = torch.tensor(bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))))
        starts = [a for a, b in offsets]; ends = [b for a, b in offsets]
        res = []
        for s, e, lab in decode_spans(text, offsets, pt.tolist()):
            tk = [i for i in range(bisect_right(ends, s), bisect_left(starts, e)) if ends[i] > starts[i]]
            res.append((int(s), int(e), lab, float(probs[tk, pt[tk]].mean()) if tk else 0.0))
        preds[doc_id] = res
    return preds

def apply_thr(raw, thr):
    return {d: [(s, e, l) for s, e, l, c in sp if c >= thr.get(l, 0.0)] for d, sp in raw.items()}

def as_scored(p): return {d: [(s, e, l, 1.0) for s, e, l in sp] for d, sp in p.items()}

def merge_policy(texts, champ_filtered, policy):
    out = {}
    for d, text in texts.items():
        rules = counsel_spans(text, policy)
        keep = [k for k in champ_filtered[d] if k[2] not in (A_, R_)]
        keep = [k for k in keep if not any(k[0] < re_ and k[1] > rs for rs, re_, _ in rules)]
        out[d] = sorted(set(keep + rules))
    return out

champion_model.eval()
c_filt = apply_thr(predict_texts(champion_model, val_texts), champ_thr)
c_macro, c_f1 = score(as_scored(c_filt), {})
assert abs(c_macro - 0.1586) < 2e-3, f"champion fold-2 macro {c_macro:.4f} != 0.1586; stop"
m_macro, m_f1 = score(as_scored(merge_policy(val_texts, c_filt, chosen)), {})
print(f"\nFOLD 2: champion {c_macro:.4f} -> champion + counsel rules (default={chosen}) {m_macro:.4f}  (delta {m_macro - c_macro:+.4f})")
print(pd.DataFrame({"champion": c_f1, "champ+rules": m_f1}).round(3).to_string())

In [ ]:
import re, random, numpy as np
from collections import Counter
random.seed(2)

LOOK = ["APP", "RESP", "WIT", "JUDGE"]
def norm(s): return re.sub(r"\s+", " ", s).strip().lower()

stats = {l: {"n": 0, "len": [], "pos": [], "prev": Counter(), "next": Counter(), "ex": []} for l in LOOK}
for d, text in train_texts.items():          # FOLD 1 ONLY
    L = len(text)
    for e in train_gold[d]:
        l = e[2]
        if l not in stats: continue
        s, t = int(e[0]), int(e[1]); st = stats[l]
        st["n"] += 1; st["len"].append(t - s); st["pos"].append(s / max(L, 1))
        left = norm(text[max(0, s - 40):s]).split(); right = norm(text[t:t + 40]).split()
        st["prev"][" ".join(left[-2:]) or "<bos>"] += 1
        st["next"][" ".join(right[:2]) or "<eos>"] += 1
        st["ex"].append((text[max(0, s - 60):s].replace("\n", " "),
                         text[s:t].replace("\n", " "),
                         text[t:t + 40].replace("\n", " ")))

for l in LOOK:
    st = stats[l]; ln = np.array(st["len"]); ps = np.array(st["pos"])
    print(f"\n===== {l}: n={st['n']}  chars median={np.median(ln):.0f} p90={np.percentile(ln, 90):.0f}  "
          f"doc-position median={np.median(ps):.2f}  share in first 5% of doc={np.mean(ps < 0.05):.2f}")
    print("  prev2:", st["prev"].most_common(8))
    print("  next2:", st["next"].most_common(8))
    for a, b, c in random.sample(st["ex"], min(5, len(st["ex"]))):
        print(f"   ...{a!r} [[{b[:80]!r}]] {c!r}")

In [ ]:
import re, random
random.seed(3)

TOK = r"(?:[A-Z]\.(?:\s?[A-Z]\.)*|[A-Z][a-z']+|[A-Z]{2,})"
TOK_RE = re.compile(TOK)
NAME = rf"{TOK}(?:[ \t]+{TOK}){{0,3}}"
STOP = {"JUDGMENT","JUDGEMENT","ORDER","BENCH","CORAM","BEFORE","PRESENT","HON'BLE","MR","MRS","MS",
        "JUSTICE","THE","AND","OF","DATE","NEW","DELHI","BY","DELIVERED"}

def back_name(text, end, max_toks=4):
    ws = max(0, end - 80)
    toks = [(m.start() + ws, m.end() + ws) for m in TOK_RE.finditer(text[ws:end])]
    run = []
    for a, b in reversed(toks):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t") != "": break
        if text[a:b].upper().strip(".") in STOP: break
        run.insert(0, (a, b))
        if len(run) == max_toks: break
    return (run[0][0], run[-1][1]) if run else None

def trim(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"): s += 1
    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"): e -= 1
    return (s, e) if e > s else None

A_RE = re.compile(rf"\bJustice[ \t]+({NAME})")
B_RE = re.compile(r",[ \t]*J{1,2}\b\.?")
C_RE = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE = re.compile(r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)[ \t]*\(J\)")

def judge_patterns(text):
    out = {"A Justice NAME": set(), "B NAME, J.": set(), "C J.(NAME)": set(), "D BENCH: X, Y (J)": set()}
    for m in A_RE.finditer(text):
        sp = trim(text, m.start(1), m.end(1)); sp and out["A Justice NAME"].add(sp)
    for m in B_RE.finditer(text):
        bn = back_name(text, m.start())
        if bn:
            sp = trim(text, *bn); sp and out["B NAME, J."].add(sp)
    for m in C_RE.finditer(text):
        inner = m.group(1)
        if re.fullmatch(r"[A-Za-z\.' ,\-]+", inner) and inner.strip().upper() not in {"J", "JJ"}:
            sp = trim(text, m.start(1), m.end(1)); sp and out["C J.(NAME)"].add(sp)
    for m in D_RE.finditer(text):
        sp = trim(text, m.start(1), m.end(1)); sp and out["D BENCH: X, Y (J)"].add(sp)
    return out

def eval_judge(texts, gold, show=4):
    names = list(judge_patterns("x").keys())
    G = {(d, int(e[0]), int(e[1])) for d, ents in gold.items() for e in ents if e[2] == "JUDGE"}
    P = {n: set() for n in names}
    for d, text in texts.items():
        for n, sp in judge_patterns(text).items():
            P[n] |= {(d, s, e) for s, e in sp}
    P["UNION"] = set().union(*[P[n] for n in names])
    print(f"gold JUDGE spans: {len(G)}\n{'pattern':22s}{'pred':>6}{'TP':>5}{'P':>7}{'R':>7}{'F1':>7}")
    for n in names + ["UNION"]:
        tp = len(G & P[n]); p = tp / max(len(P[n]), 1); r = tp / max(len(G), 1)
        print(f"{n:22s}{len(P[n]):>6}{tp:>5}{p:>7.3f}{r:>7.3f}{(2*p*r/(p+r) if p+r else 0):>7.3f}")
    cov = sum(1 for (d, s, e) in G if any(pd_ == d and ps < e and pe > s for pd_, ps, pe in P["UNION"]))
    print(f"\nboundary-agnostic coverage of gold by union: {cov}/{len(G)} = {cov/len(G):.2f}  "
          f"(exact: {len(G & P['UNION'])/len(G):.2f})")
    for n in names:
        fps = sorted(P[n] - G)
        print(f"\n--- {n}: false positives (showing {min(show, len(fps))} of {len(fps)}) ---")
        for d, s, e in random.sample(fps, min(show, len(fps))):
            t = texts[d]; print(f"   ...{t[max(0,s-50):s]!r} [[{t[s:e]!r}]] {t[e:e+25]!r}")
    miss = sorted(G - P["UNION"])
    print(f"\n--- missed gold JUDGE (showing {min(8, len(miss))} of {len(miss)}) ---")
    for d, s, e in random.sample(miss, min(8, len(miss))):
        t = texts[d]; print(f"   ...{t[max(0,s-50):s]!r} [[{t[s:e]!r}]] {t[e:e+25]!r}")

eval_judge(train_texts, train_gold)   # FOLD 1 ONLY

In [ ]:
import re, random
random.seed(4)

CAP   = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
INIT  = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
WORD  = rf"(?:{INIT}|{CAP})"
NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"
TOK_RE = re.compile(WORD)                      # back_name() reads this global

AJ_RE = re.compile(rf"(?i:\bjustice)[ \t]+({NAME2})")
B_RE  = re.compile(r",[ \t]*J{1,2}\b\.?")
B2_RE = re.compile(r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|held\b|was\b|said\b|while\b|who\b|delivered|dissent|agreed|also\b|noted|stated))")
C_RE  = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE  = re.compile(r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)[ \t]*\(J\)")
E_RE  = re.compile(r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]")
F_RE  = re.compile(r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)(?=[ \t]*(?:JUDG(?:E)?MENT|CITATION|ACT\b|\n|$))")
NAMES = ["A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)", "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"]

def judge_patterns2(text):
    out = {n: set() for n in NAMES}
    def add(n, s, e):
        sp = trim(text, s, e)
        if sp: out[n].add(sp)
    for m in AJ_RE.finditer(text): add(NAMES[0], m.start(1), m.end(1))
    for rx, n in [(B_RE, NAMES[1]), (B2_RE, NAMES[2])]:
        for m in rx.finditer(text):
            bn = back_name(text, m.start())
            if bn: add(n, *bn)
    for rx, n in [(C_RE, NAMES[3]), (E_RE, NAMES[5])]:
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if inner and inner[0].isupper() and inner.upper() not in {"J", "JJ"} and re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner):
                add(n, m.start(1), m.end(1))
    for m in D_RE.finditer(text): add(NAMES[4], m.start(1), m.end(1))
    for m in F_RE.finditer(text):
        seg, base, pos = m.group(1), m.start(1), 0
        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", seg):
            k = seg.find(part, pos)
            if k < 0: continue
            pos = k + len(part)
            part2 = re.sub(r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part)
            sp = trim(text, base + k, base + k + len(part2))
            if sp and re.fullmatch(NAME2, text[sp[0]:sp[1]]): out[NAMES[6]].add(sp)
    return out

def resolve(spans):
    out = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1] - x[0]))):
        if out and s < out[-1][1]: continue
        out.append((s, e))
    return out

# ---- 1. FOLD 1: per-pattern stats and pre-declared selection (P >= 0.4 and TP >= 5) ----
G = {(d, int(e[0]), int(e[1])) for d, ents in train_gold.items() for e in ents if e[2] == "JUDGE"}
Pn = {n: set() for n in NAMES}
for d, text in train_texts.items():
    for n, sp in judge_patterns2(text).items(): Pn[n] |= {(d, s, e) for s, e in sp}
print(f"gold JUDGE (fold 1): {len(G)}\n{'pattern':24s}{'pred':>6}{'TP':>5}{'P':>7}{'R':>7}")
keep = []
for n in NAMES:
    tp = len(G & Pn[n]); p = tp / max(len(Pn[n]), 1)
    print(f"{n:24s}{len(Pn[n]):>6}{tp:>5}{p:>7.3f}{tp / len(G):>7.3f}")
    if p >= 0.4 and tp >= 5: keep.append(n)
print("kept patterns:", keep)

def judge_spans_doc(text, keep):
    allsp = set()
    for n, sp in judge_patterns2(text).items():
        if n in keep: allsp |= sp
    return resolve(allsp)

U1 = {(d, s, e) for d, text in train_texts.items() for s, e in judge_spans_doc(text, keep)}
tp = len(G & U1); p = tp / max(len(U1), 1); r = tp / len(G)
print(f"fold 1 union of kept (overlaps resolved): pred={len(U1)} TP={tp} P={p:.3f} R={r:.3f} F1={2*p*r/(p+r):.3f}")

# ---- 2. FOLD 2, once: champion + counsel rules + JUDGE rules (champion JUDGE spans replaced) ----
assert "c_filt" in globals(), "re-run the earlier fold-2 cell to define c_filt"
def merge_all(texts, champ_filtered, keep):
    out = {}
    for d, text in texts.items():
        rules = list(counsel_spans(text, chosen)) + [(s, e, "JUDGE") for s, e in judge_spans_doc(text, keep)]
        kept = [k for k in champ_filtered[d] if k[2] not in (A_, R_, "JUDGE")]
        kept = [k for k in kept if not any(k[0] < re_ and k[1] > rs for rs, re_, _ in rules)]
        out[d] = sorted(set(kept + rules))
    return out

base_macro, base_f1 = score(as_scored(merge_policy(val_texts, c_filt, chosen)), {})
new_macro, new_f1 = score(as_scored(merge_all(val_texts, c_filt, keep)), {})
print(f"\nFOLD 2: champion {c_macro:.4f} | +counsel {base_macro:.4f} | +counsel+JUDGE {new_macro:.4f}  (JUDGE step: {new_macro - base_macro:+.4f})")
print(pd.DataFrame({"champ+counsel": base_f1, "champ+counsel+judge": new_f1}).round(3).to_string())

In [ ]:
import json, os
from collections import Counter

FOLD3 = "/kaggle/working/cs781_backup_download/phase2_lner/Phase_2_LNER/Test/fold3-text.json"
NEW_PATH = "/kaggle/working/prediction_champion_plus_rules.json"
assert not os.path.exists(NEW_PATH), "refusing to overwrite an existing file"

with open(FOLD3, encoding="utf-8") as f: test_texts = json.load(f)
assert isinstance(test_texts, dict), f"fold3 file is {type(test_texts)}"
print("fold 3 docs:", len(test_texts))

champion_model.eval()
test_filt = apply_thr(predict_texts(champion_model, test_texts), champ_thr)
test_merged = merge_all(test_texts, test_filt, keep)
new_obj = {str(d): [[int(s), int(e), l] for s, e, l in sp] for d, sp in test_merged.items()}

def validate(obj, texts, name):
    T = {str(k): v for k, v in texts.items()}
    issues, labs = Counter(), Counter()
    for d, sp in obj.items():
        text = T.get(str(d))
        if text is None: issues["doc not in fold3"] += 1; continue
        for x in sp:
            if len(x) != 3: issues["span not length 3"] += 1; continue
            s, e, l = x
            if l not in ENT: issues["unknown label"] += 1
            if not (isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= len(text)):
                issues["bad offsets"] += 1; continue
            if text[s:e] != text[s:e].strip(): issues["whitespace edge"] += 1
            labs[l] += 1
        ss = sorted((x[0], x[1]) for x in sp if len(x) == 3)
        issues["overlapping pairs"] += sum(1 for a, b in zip(ss, ss[1:]) if b[0] < a[1])
    issues["missing docs"] = len(set(T) - set(map(str, obj)))
    issues = {k: v for k, v in issues.items() if v}
    print(f"\n[{name}] docs={len(obj)} spans={sum(labs.values())} issues={issues or 'none'}")
    print("  per-label:", dict(sorted(labs.items())))
    return issues

old_path = "/kaggle/working/prediction.json"
if os.path.exists(old_path):
    with open(old_path, encoding="utf-8") as f: old = json.load(f)
    print("existing prediction.json: top-level", type(old).__name__, "len", len(old))
    if isinstance(old, dict) and old:
        k = next(iter(old)); print("  sample key:", repr(k), "| sample value:", old[k][:3])
        validate(old, test_texts, "existing prediction.json (read-only)")

issues = validate(new_obj, test_texts, "NEW candidate")
fold2_merged = merge_all(val_texts, c_filt, keep)
print(f"\nspans/doc: fold 2 merged = {sum(map(len, fold2_merged.values()))/len(val_texts):.1f}, "
      f"fold 3 merged = {sum(map(len, new_obj.values()))/len(test_texts):.1f}")

hard = {"bad offsets", "unknown label", "span not length 3", "missing docs", "doc not in fold3"}
if hard & set(issues):
    print("\nNOT WRITTEN: hard format errors:", hard & set(issues))
else:
    with open(NEW_PATH, "x", encoding="utf-8") as f: json.dump(new_obj, f)
    print("\nWrote:", NEW_PATH)